# Классификация ботов - итоговое решение

## Установка обязательных зависимостей

Перед запуском notebook должны выполняться два требования:

- рядом с `solution.ipynb` должен находиться неизменённый файл `metric.py` с официальной функцией `precision_at_recall`;
- должны быть установлены библиотеки с версиями из `requirements.txt`.

Следующая ячейка устанавливает те же версии библиотек явным списком. В обычном окружении их также можно установить командой `pip install -r requirements.txt`.

In [1]:
# Устанавливаем точные версии библиотек для воспроизводимого запуска.
%pip install --disable-pip-version-check numpy==2.5.3 pandas==3.0.6 scipy==1.18.1 scikit-learn==1.9.1 lightgbm==4.7.0 catboost==1.2.10 xgboost==3.4.1 optuna==5.0.0 matplotlib==3.11.2 seaborn==0.13.2 jupyter==1.1.1 ipykernel==7.3.0

Note: you may need to restart the kernel to use updated packages.


## 1. Исследование данных

In [2]:
# Импортируем стандартные и табличные библиотеки.
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

# Подключаем официальную функцию основной метрики.
from metric import precision_at_recall

# Фиксируем общие параметры эксперимента и путь к данным.
RANDOM_STATE = 42
TARGET_RECALL = 0.70
DATA_DIR = Path('data')

# Настраиваем отображение таблиц и скрываем служебные предупреждения.
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)
warnings.filterwarnings('ignore')

# Загружаем train, test и events с разбором временных колонок.
date_cols = ['cookie_created_at', 'window_start_ts', 'window_end_ts']
train = pd.read_csv(DATA_DIR / 'train.csv', parse_dates=date_cols)
test = pd.read_csv(DATA_DIR / 'test.csv', parse_dates=date_cols)
events = pd.read_csv(DATA_DIR / 'events.csv.gz', parse_dates=['event_ts'])

In [3]:
def compact_overview(df, name):
    """Выводит размер таблицы, типы колонок, пропуски, уникальные значения и дубликаты."""
    # Показываем размер таблицы перед подробной статистикой.
    print()

    print(f'{name}: shape={df.shape}')
    summary = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing_n': df.isna().sum(),
        'missing_share': df.isna().mean(),
        'num_unique': df.nunique(dropna=False),
    })
    display(summary)

    print('exact duplicate rows:', int(df.duplicated().sum()))


for frame, name in [(train, 'train'), (test, 'test'), (events, 'events')]:
    compact_overview(frame, name)

# Проверяем распределение целевого класса.
target_counts = train['target'].value_counts(dropna=False).sort_index()
print()
print('target counts:')
print(target_counts.to_string())
print('positive share:', float(train['target'].mean()))
# Проверяем уникальность cookie и отсутствие пересечения train с test.
print('train cookie_id duplicates:', int(train['cookie_id'].duplicated().sum()))
print('test cookie_id duplicates:', int(test['cookie_id'].duplicated().sum()))
overlap = set(train['cookie_id']).intersection(test['cookie_id'])
print('train/test cookie_id overlap:', len(overlap))


train: shape=(11091, 5)


,dtype,missing_n,missing_share,num_unique
cookie_id,str,0,0.0,11091
cookie_created_at,datetime64[us],0,0.0,11085
window_start_ts,datetime64[us],0,0.0,14
window_end_ts,datetime64[us],0,0.0,14
target,int64,0,0.0,2


exact duplicate rows: 0

test: shape=(4909, 4)


,dtype,missing_n,missing_share,num_unique
cookie_id,str,0,0.0,4909
cookie_created_at,datetime64[us],0,0.0,4908
window_start_ts,datetime64[us],0,0.0,7
window_end_ts,datetime64[us],0,0.0,7


exact duplicate rows: 0

events: shape=(328905, 14)


,dtype,missing_n,missing_share,num_unique
cookie_id,str,0,0.000000,16000
event_ts,datetime64[us],0,0.000000,285094
eid,int64,0,0.000000,10
event_name,str,0,0.000000,10
platform,str,0,0.000000,11
user_agent,str,0,0.000000,148
item_id,float64,114597,0.348420,53857
item_category,str,32920,0.100090,16
item_location,str,23793,0.072340,41
seller_type,str,133853,0.406966,3


exact duplicate rows: 4863

target counts:
target
0    10192
1      899
positive share: 0.0810567126498963
train cookie_id duplicates: 0
test cookie_id duplicates: 0
train/test cookie_id overlap: 0


### Точные дубликаты в `events`

В `train` и `test` точных дубликатов строк нет, а в `events` обнаружено 4863 лишние строки, полностью совпадающие с одной из предыдущих строк по всем 14 колонкам. Это не просто одинаковые названия действий: у копий совпадают `cookie_id`, точное время `event_ts`, тип события, платформа, User-Agent и все параметры события.

Такие повторы могут появляться как из-за повторного действия пользователя, так и из-за особенностей сбора данных. Поэтому они не удаляются безусловно: при feature engineering отдельно измеряется доля дубликатов, а временные характеристики рассчитываются как по исходной последовательности, так и после удаления точных копий.

In [4]:
# Собираем количество и долю пропусков для каждой колонки всех таблиц.
missing_rows = []
for table_name, frame in [('train', train), ('test', test), ('events', events)]:
    for column in frame.columns:
        missing_rows.append({
            'table': table_name,
            'column': column,
            'missing_n': int(frame[column].isna().sum()),
            'missing_share': float(frame[column].isna().mean()),
        })

# Выводим только колонки, в которых присутствуют пропуски.
missing_summary = pd.DataFrame(missing_rows)
display(
    missing_summary.loc[missing_summary['missing_n'].gt(0)]
    .sort_values(['table', 'missing_share'], ascending=[True, False])
    .reset_index(drop=True)
)

# Проверяем, пропускаются ли связанные поля поиска и pointer одновременно.
paired_missing = pd.DataFrame({
    'field_pair': ['search_query + search_page', 'pointer_x + pointer_y'],
    'both_missing': [
        int((events['search_query'].isna() & events['search_page'].isna()).sum()),
        int((events['pointer_x'].isna() & events['pointer_y'].isna()).sum()),
    ],
    'only_one_missing': [
        int(events['search_query'].isna().ne(events['search_page'].isna()).sum()),
        int(events['pointer_x'].isna().ne(events['pointer_y'].isna()).sum()),
    ],
})
display(paired_missing)

,table,column,missing_n,missing_share
0,events,search_query,228503,0.694739
1,events,search_page,228503,0.694739
2,events,pointer_x,220365,0.669996
3,events,pointer_y,220365,0.669996
4,events,seller_type,133853,0.406966
5,events,item_id,114597,0.348420
6,events,item_category,32920,0.100090
7,events,item_location,23793,0.072340


,field_pair,both_missing,only_one_missing
0,search_query + search_page,228503,0
1,pointer_x + pointer_y,220365,0


### Пропуски в `events`

В обязательных полях события - `cookie_id`, `event_ts`, `eid`, `event_name`, `platform` и `user_agent` - пропусков нет. Значит, каждое событие можно связать с cookie, расположить во времени и определить его тип и технический контекст.

Пропуски в остальных колонках в основном структурные, а не случайные:

- `item_id` отсутствует примерно в 34.8% событий, потому что не каждое действие связано с конкретным объявлением;
- `seller_type` отсутствует примерно в 40.7% событий и имеет смысл только там, где известен продавец;
- `search_query` и `search_page` одновременно отсутствуют примерно в 69.5% строк: эти поля относятся прежде всего к поисковым событиям;
- `pointer_x` и `pointer_y` одновременно отсутствуют примерно в 67.0% строк: координаты записываются только для части пользовательских действий и платформ;
- у `item_category` и `item_location` пропусков заметно меньше - около 10.0% и 7.2% соответственно.

Поэтому отсутствие значения само может описывать поведение cookie. Например, доля событий с координатами указателя может отличать обычное взаимодействие с интерфейсом от автоматизированных запросов. Пропуски не заполняются произвольным средним значением до агрегации. Вместо этого на уровне cookie строятся `count`, `present_share`, `nunique` и статистики доступных значений; если статистику вычислить невозможно, после агрегации остаётся `NaN`, который древесные модели обрабатывают отдельно. Для логистической регрессии такие значения заполняются внутри preprocessing-пайплайна. Все признаки наличия данных рассчитываются только по событиям внутри окна наблюдения, поэтому сами по себе утечку не создают.

In [5]:
# Изучаем частоты событий и соответствие между eid и event_name.
print('event_name counts:')
display(events['event_name'].value_counts(dropna=False).to_frame('count'))
print('eid -> event_name mapping:')
display(events.groupby('eid')['event_name'].agg(nunique='nunique', values=lambda x: sorted(x.unique())).sort_index())
# Сравниваем исходные и приведённые к нижнему регистру значения platform.
print('platform raw counts:')
display(events['platform'].value_counts(dropna=False).to_frame('count'))
print('platform lower-case counts:')
display(events['platform'].str.lower().value_counts(dropna=False).to_frame('count'))

# Выводим распределения низкокардинальных категориальных колонок.
low_card_cols = ['item_category', 'item_location', 'seller_type']
for col in low_card_cols:
    print()
    print(f'{col} counts:')
    display(events[col].value_counts(dropna=False).head(50).to_frame('count'))

# Смотрим самые частые строки User-Agent.
print()
print('Top user_agent values:')
display(events['user_agent'].value_counts().head(20).to_frame('count'))

event_name counts:


,count
event_name,
item_view,120817
search_results_view,100402
photo_swipe,36517
favorite_add,19049
seller_page_view,17403
contact_phone_show,11316
captcha_shown,7928
login,6267
contact_chat_open,6125


eid -> event_name mapping:


,nunique,values
eid,,
100,1,[search_results_view]
200,1,[item_view]
210,1,[photo_swipe]
220,1,[seller_page_view]
300,1,[contact_phone_show]
301,1,[contact_chat_open]
303,1,[contact_message_sent]
400,1,[favorite_add]
500,1,[login]


platform raw counts:


...

,count
item_location,
sankt-peterburg,39221
novosibirsk,37989
moskva,37964
ekaterinburg,37281
NaN,23793
ryazan,4627
izhevsk,4576
barnaul,4559
astrahan,4511



seller_type counts:


,count
seller_type,
NaN,133853
private,112631
pro,82421



Top user_agent values:


,count
user_agent,
"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 YaBrowser/23.11.0.0 Safari/537.36",3737
Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:119.0) Gecko/20100101 Firefox/119.0,3670
"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 YaBrowser/23.9.0.0 Safari/537.36",3664
"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 YaBrowser/23.0.0.0 Safari/537.36",3640
"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 YaBrowser/23.3.0.0 Safari/537.36",3558
"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",3481
"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",3470
"Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/118.0.0.0 Safari/537.36",3462
"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 YaBrowser/23.8.0.0 Safari/537.36",3461


### Что полезного содержится в `user_agent` и почему строку нельзя брать как есть

`user_agent` - техническая строка, которой клиент представляется серверу. Например, запись `Mozilla/5.0 (X11; Linux x86_64) ... HeadlessChrome/118.0.0.0 ...` содержит сразу несколько частей:

- операционную систему и устройство: `Windows NT`, `Macintosh`, `Linux`, Android, iPhone и другие;
- семейство браузера: Chrome, Firefox, `YaBrowser` и другие;
- движок браузера: AppleWebKit, Gecko;
- номера версий браузера и операционной системы;
- указания на мобильное устройство;
- иногда явные технические маркеры автоматизации, например `HeadlessChrome`, `bot`, `python`, `requests`, `curl`, `wget`, `scrapy`, `selenium` или `playwright`.

Сырую строку не следует напрямую передавать табличной модели по нескольким причинам:

1. В данных 148 точных вариантов, причём строки одного семейства различаются только номером версии. При one-hot encoding они стали бы независимыми категориями.
2. Train и test разделены по времени. Новая версия браузера может впервые появиться в test, поэтому запоминание точной строки плохо переносится на будущие даты.
3. Label encoding создал бы искусственный порядок между строками, хотя браузеры не имеют естественной числовой шкалы.
4. Одна и та же строка повторяется во многих событиях cookie. Для классификации cookie полезнее описать устойчивость и состав User-Agent на всём окне.
5. User-Agent можно подменить. Даже явный `HeadlessChrome` является сильным сигналом, но не доказательством класса сам по себе.

Поэтому из `user_agent` строятся агрегированные признаки на уровне cookie:

- `user_agent_dist_nunique` и `user_agent_dist_nunique_ratio` - сколько разных строк использовалось и как часто они менялись;
- `user_agent_dist_top_share` - доля наиболее частого User-Agent;
- `user_agent_dist_entropy` и `user_agent_dist_entropy_norm` - разнообразие User-Agent;
- `ua_headless_share` и `ua_automation_share` - доля событий с явными маркерами headless-режима или автоматизации;
- `ua_mobile_share`, `ua_linux_share`, `ua_windows_share`, `ua_macos_share` - доли событий для основных семейств устройств и ОС.

In [6]:
# Выводим наиболее частые поисковые запросы с учётом пропусков.
print()
print('Top search_query values:')
display(events['search_query'].value_counts(dropna=False).head(20).to_frame('count'))


Top search_query values:


,count
search_query,
NaN,228503
toyota camry,919
кофемашина,919
клининг,918
renault logan,916
kia rio 2019,908
посудомоечная машина,904
вторичка 2к,904
шкаф купе,897


In [7]:
# Объединяем границы окон train и test в одну таблицу метаданных.
meta = pd.concat([
    train[['cookie_id', 'window_start_ts', 'window_end_ts']].assign(split='train'),
    test[['cookie_id', 'window_start_ts', 'window_end_ts']].assign(split='test'),
], ignore_index=True)
assert not meta['cookie_id'].duplicated().any()

# Присоединяем к каждому событию границы его окна.
evm = events.merge(meta, on='cookie_id', how='left', validate='many_to_one', indicator=True)
print('events with cookie absent from train/test:', int(evm['_merge'].ne('both').sum()))
# Помечаем положение каждого события относительно допустимого окна.
evm['before_window'] = evm['event_ts'] < evm['window_start_ts']
evm['at_or_after_end'] = evm['event_ts'] >= evm['window_end_ts']
evm['in_window'] = (~evm['before_window']) & (~evm['at_or_after_end']) & evm['_merge'].eq('both')

# Считаем события внутри и вне окна отдельно для train и test.
window_status = evm.groupby(['split'], dropna=False).agg(
    events_total=('cookie_id', 'size'),
    in_window=('in_window', 'sum'),
    before_window=('before_window', 'sum'),
    at_or_after_end=('at_or_after_end', 'sum'),
)
display(window_status)
print('events exactly at window_start:', int((evm['event_ts'] == evm['window_start_ts']).sum()))
print('events exactly at window_end:', int((evm['event_ts'] == evm['window_end_ts']).sum()))

# Оставляем допустимые события и сравниваем объёмы до и после фильтрации.
ev_window = evm.loc[evm['in_window'], events.columns].copy()
counts_raw = events.groupby('cookie_id').size()
counts_win = ev_window.groupby('cookie_id').size()
cookie_event_summary = meta[['cookie_id', 'split']].copy()
cookie_event_summary['events_raw'] = cookie_event_summary['cookie_id'].map(counts_raw).fillna(0).astype(int)
cookie_event_summary['events_in_window'] = cookie_event_summary['cookie_id'].map(counts_win).fillna(0).astype(int)
for split, part in cookie_event_summary.groupby('split'):
    print()
    print(f'{split}: cookies without any raw events={int((part.events_raw == 0).sum())}; without in-window events={int((part.events_in_window == 0).sum())}')
    display(part[['events_raw', 'events_in_window']].describe(percentiles=[.01, .1, .25, .5, .75, .9, .95, .99]).T)

# Анализируем точные дубликаты только среди допустимых событий.
print('exact duplicate rows, in-window events:', int(ev_window.duplicated().sum()))
dups_by_type = ev_window.loc[ev_window.duplicated(keep=False)].groupby('event_name').size().sort_values(ascending=False)
display(dups_by_type.to_frame('rows_in_duplicate_groups'))

events with cookie absent from train/test: 0


,events_total,in_window,before_window,at_or_after_end
split,,,,
test,89690,89690,0,0
train,239215,198436,0,40779


events exactly at window_start: 0
events exactly at window_end: 68

test: cookies without any raw events=0; without in-window events=0


,count,mean,std,min,1%,10%,25%,50%,75%,90%,95%,99%,max
events_raw,4909.0,18.270524,18.658159,1.0,1.0,3.0,6.0,12.0,23.0,40.0,56.0,92.0,191.0
events_in_window,4909.0,18.270524,18.658159,1.0,1.0,3.0,6.0,12.0,23.0,40.0,56.0,92.0,191.0



train: cookies without any raw events=0; without in-window events=0


,count,mean,std,min,1%,10%,25%,50%,75%,90%,95%,99%,max
events_raw,11091.0,21.568389,20.065333,1.0,1.0,5.0,8.0,15.0,28.0,48.0,61.0,97.0,191.0
events_in_window,11091.0,17.891624,17.706829,1.0,1.0,3.0,6.0,12.0,23.0,40.0,53.0,87.0,169.0


exact duplicate rows, in-window events: 4293


,rows_in_duplicate_groups
event_name,
item_view,3276
search_results_view,2638
photo_swipe,1028
favorite_add,568
seller_page_view,398
contact_phone_show,286
login,164
contact_chat_open,138
contact_message_sent,90


### Повторы после ограничения окна

После фильтрации остаются только события из полуоткрытого окна `window_start_ts <= event_ts < window_end_ts`. В нём находится **4293 лишние точные копии**; 570 копий за пределами окна в признаки не попадают. Таблица по `event_name` построена с `duplicated(keep=False)`, поэтому она показывает всех участников повторяющихся групп - первую запись и её копии.

In [8]:
def time_summary(df, name):
    """Показывает временные диапазоны и проверяет корректность окон наблюдения."""
    # Находим минимальные и максимальные значения основных временных полей.
    out = df.agg({
        'cookie_created_at': ['min', 'max'],
        'window_start_ts': ['min', 'max'],
        'window_end_ts': ['min', 'max'],
    })
    print()
    print(f'{name}: временные диапазоны:')
    display(out)
    # Проверяем длину окон и возможные противоречия во времени создания cookie.
    print('распределение длины окна (в часах):')
    display(((df.window_end_ts - df.window_start_ts).dt.total_seconds() / 3600).value_counts().sort_index().to_frame('количество cookie'))
    print('cookie, созданные после начала окна:', int((df.cookie_created_at > df.window_start_ts).sum()))
    print('cookie, созданные после или в момент конца окна:', int((df.cookie_created_at >= df.window_end_ts).sum()))

# Сравниваем временные характеристики train и test.
time_summary(train, 'train')
time_summary(test, 'test')
print()
print('диапазон event_ts:', events['event_ts'].min(), '—', events['event_ts'].max())

# Считаем число объектов и долю положительного класса по дням.
daily_train = train.assign(window_date=train.window_start_ts.dt.date).groupby('window_date').agg(
    количество=('target', 'size'),
    положительных=('target', 'sum'),
    доля_положительных=('target', 'mean'),
)
daily_test = test.assign(window_date=test.window_start_ts.dt.date).groupby('window_date').size().to_frame('количество')
print()
print('train по датам окна:')
display(daily_train)
print('test по датам окна:')
display(daily_test)

# Готовим проверки покрытия временного диапазона событий.
all_window_dates = set(pd.concat([train.window_start_ts, test.window_start_ts], ignore_index=True).dt.normalize())
event_dates = set(events.event_ts.dt.normalize())
in_window_rows = evm.loc[evm['in_window']]
filtered_rows_valid = bool(
    in_window_rows.event_ts.ge(in_window_rows.window_start_ts).all()
    and in_window_rows.event_ts.lt(in_window_rows.window_end_ts).all()
)

# Выводим результаты проверок временной структуры.
print()
print('train отсортирован по window_start_ts:', train.window_start_ts.is_monotonic_increasing)
print('test отсортирован по window_start_ts:', test.window_start_ts.is_monotonic_increasing)
print('events покрывает все даты окон:', all_window_dates.issubset(event_dates))
print('отфильтрованные события находятся внутри окон своих cookie:', filtered_rows_valid)
print('событий до начала окна:', int(evm.before_window.sum()))
print('событий в момент или после конца окна:', int(evm.at_or_after_end.sum()))


train: временные диапазоны:


,cookie_created_at,window_start_ts,window_end_ts
min,2025-01-22 17:09:47,2026-04-06,2026-04-07
max,2026-04-18 22:45:04,2026-04-19,2026-04-20


распределение длины окна (в часах):


,количество cookie
24.0,11091


cookie, созданные после начала окна: 0
cookie, созданные после или в момент конца окна: 0

test: временные диапазоны:


...

cookie, созданные после начала окна: 0
cookie, созданные после или в момент конца окна: 0

диапазон event_ts: 2026-04-06 00:02:57 — 2026-04-26 23:59:50

train по датам окна:


,количество,положительных,доля_положительных
window_date,,,
2026-04-06,772,68,0.088083
2026-04-07,797,59,0.074028
2026-04-08,834,70,0.083933
2026-04-09,902,66,0.073171
2026-04-10,912,82,0.089912
2026-04-11,896,79,0.088170
2026-04-12,817,67,0.082007
2026-04-13,843,53,0.062871
2026-04-14,826,70,0.084746


test по датам окна:


,количество
window_date,
2026-04-20,652
2026-04-21,628
2026-04-22,637
2026-04-23,660
2026-04-24,760
2026-04-25,759
2026-04-26,813



train отсортирован по window_start_ts: True
test отсортирован по window_start_ts: True
events покрывает все даты окон: True
отфильтрованные события находятся внутри окон своих cookie: True
событий до начала окна: 0
событий в момент или после конца окна: 40779


### Выводы из анализа времени и порядка строк

1. **Все окна имеют одинаковую длину - 24 часа.** Поэтому количество событий и другие признаки, связанные с их числом, можно сравнивать между cookie напрямую. Если бы окна имели разную длину, такие признаки пришлось бы нормировать, например считать количество событий в час.

2. **Все cookie созданы до начала своего окна.** Поэтому при вычислении возраста cookie не должно возникать отрицательных значений.

3. **Test расположен позже train.** В `train` находятся данные за 6-19 апреля 2026 года, а в `test` - за 20-26 апреля. Поэтому для валидации я использовал временное разбиение: модель обучается на более ранних данных и проверяется на более поздних. Такой подход лучше соответствует тому, как устроена тестовая выборка. Случайный `StratifiedKFold` здесь менее подходящий, так как при нём данные из разных дат перемешиваются.

Используются три последовательных временных разбиения с расширяющейся обучающей выборкой и без перемешивания:

a) train 06-10 апреля, validation 11-13 апреля

b) train 06-13 апреля, validation 14-16 апреля

c) train 06-16 апреля, validation 17-19 апреля

4. **Доля положительного класса меняется по дням.** В `train` она находится примерно в диапазоне от 6.3% до 9.3%. Поэтому при временной валидации лучше проверять модель на нескольких последовательных временных промежутках, а не только на одном дне.

5. **Строки train и test уже отсортированы по началу окна.** При этом в дальнейшем для разбиения я всё равно использую сами даты, а не номера строк. Благодаря этому изменение порядка строк в DataFrame не повлияет на временное разбиение.

6. **Общий диапазон `event_ts` покрывает даты train и test.** При этом это ещё не означает, что каждое событие относится к нужному суточному окну. Поэтому при построении признаков используются только события, удовлетворяющие условию `window_start_ts <= event_ts < window_end_ts`.

In [9]:
# Готовим train-события и вспомогательные поля для предварительного анализа признаков.
ev_train = ev_window.merge(train[['cookie_id', 'target']], on='cookie_id', how='inner', validate='many_to_one')
ev_train = ev_train.sort_values(['cookie_id', 'event_ts'], kind='mergesort').copy()
ev_train['gap_s'] = ev_train.groupby('cookie_id')['event_ts'].diff().dt.total_seconds()
ev_train['platform_norm'] = ev_train['platform'].str.lower().replace({'iphone': 'ios'})
ev_train['minute'] = ev_train['event_ts'].dt.floor('min')
ev_train['hour'] = ev_train['event_ts'].dt.floor('h')
ev_train['is_exact_duplicate'] = ev_train[events.columns].duplicated(keep=False)
ev_train['ua_headless'] = ev_train['user_agent'].str.contains('headless', case=False, na=False)
ev_train['ua_automation_token'] = ev_train['user_agent'].str.contains('bot|python|requests|curl|wget|scrapy|selenium|playwright', case=False, regex=True, na=False)

# Создаём таблицу probe с одной строкой на cookie.
g = ev_train.groupby('cookie_id')
probe = pd.DataFrame(index=train.cookie_id)
# Считаем объём активности и временные характеристики.
probe['total_events'] = g.size()
probe['event_type_nunique'] = g.event_name.nunique()
probe['active_duration_s'] = g.event_ts.max().sub(g.event_ts.min()).dt.total_seconds()
probe['gap_mean_s'] = g.gap_s.mean()
probe['gap_median_s'] = g.gap_s.median()
probe['gap_std_s'] = g.gap_s.std()
probe['gap_cv'] = probe['gap_std_s'] / probe['gap_mean_s'].replace(0, np.nan)
probe['zero_gap_share'] = g.gap_s.apply(lambda s: s.eq(0).mean())
# Считаем разнообразие значений и технические сигналы поведения.
probe['active_hours'] = g.hour.nunique()
probe['platform_nunique'] = g.platform_norm.nunique()
probe['ua_nunique'] = g.user_agent.nunique()
probe['item_nunique'] = g.item_id.nunique()
probe['query_nunique'] = g.search_query.nunique()
probe['duplicate_row_share'] = g.is_exact_duplicate.mean()
probe['headless_share'] = g.ua_headless.mean()
probe['automation_ua_share'] = g.ua_automation_token.mean()
probe['pointer_present_share'] = g.pointer_x.apply(lambda s: s.notna().mean())
probe['captcha_count'] = g.event_name.apply(lambda s: s.eq('captcha_shown').sum())
probe['max_events_per_minute'] = ev_train.groupby(['cookie_id', 'minute']).size().groupby(level=0).max()
# Добавляем target и вычисляем возраст cookie на начало окна.
probe = probe.reset_index().merge(train[['cookie_id', 'target', 'cookie_created_at', 'window_start_ts']], on='cookie_id')
probe['cookie_age_days'] = (probe.window_start_ts - probe.cookie_created_at).dt.total_seconds() / 86400

# Выбираем признаки для сравнения средних между классами.
summary_features = ['total_events', 'active_duration_s', 'gap_median_s', 'gap_cv', 'zero_gap_share',
                    'active_hours', 'platform_nunique', 'ua_nunique', 'item_nunique', 'query_nunique',
                    'duplicate_row_share', 'headless_share', 'automation_ua_share',
                    'pointer_present_share', 'captcha_count', 'max_events_per_minute', 'cookie_age_days']
# Сравниваем долю положительного класса при наличии отдельных маркеров.
for flag in ['headless_share', 'automation_ua_share', 'duplicate_row_share']:
    has_flag = probe[flag].gt(0)
    print(f'{flag}: cookies={int(has_flag.sum())}, positive_share={probe.loc[has_flag, "target"].mean():.4f}, no_flag_positive_share={probe.loc[~has_flag, "target"].mean():.4f}')

headless_share: cookies=286, positive_share=0.2622, no_flag_positive_share=0.0763
automation_ua_share: cookies=125, positive_share=0.2240, no_flag_positive_share=0.0794
duplicate_row_share: cookies=2360, positive_share=0.1220, no_flag_positive_share=0.0700


In [10]:
# Рассчитываем средние значения выбранных признаков отдельно для людей и ботов.
mean_by_target = probe.groupby('target')[summary_features].mean().T
mean_difference = pd.DataFrame({
    'class_0_mean': mean_by_target[0],
    'class_1_mean': mean_by_target[1],
})
# Считаем абсолютную и относительную разницу между классами.
mean_difference['class_1_minus_class_0'] = (
    mean_difference['class_1_mean'] - mean_difference['class_0_mean']
)
mean_difference['relative_difference_pct'] = np.where(
    mean_difference['class_0_mean'].ne(0),
    100 * mean_difference['class_1_minus_class_0'] / mean_difference['class_0_mean'].abs(),
    np.nan,
)
# Для долей дополнительно выражаем разницу в процентных пунктах.
share_features = [
    feature for feature in summary_features
    if feature.endswith('_share')
]
mean_difference['difference_percentage_points'] = np.where(
    mean_difference.index.isin(share_features),
    100 * mean_difference['class_1_minus_class_0'],
    np.nan,
)
# Указываем класс с более высоким средним значением признака.
mean_difference['higher_mean_class'] = np.select(
    [
        mean_difference['class_1_minus_class_0'].gt(0),
        mean_difference['class_1_minus_class_0'].lt(0),
    ],
    ['1 (bot)', '0 (human)'],
    default='equal',
)

# Сортируем признаки по величине относительного различия.
display(
    mean_difference.sort_values(
        'relative_difference_pct',
        key=lambda s: s.abs(),
        ascending=False,
    ).round(4)
)

,class_0_mean,class_1_mean,class_1_minus_class_0,relative_difference_pct,difference_percentage_points,higher_mean_class
headless_share,0.0207,0.0834,0.0627,302.9754,6.2724,1 (bot)
automation_ua_share,0.0095,0.0311,0.0216,227.2548,2.1628,1 (bot)
total_events,16.8690,29.4850,12.6160,74.7878,NaN,1 (bot)
max_events_per_minute,2.5334,4.3259,1.7926,70.7581,NaN,1 (bot)
item_nunique,8.9172,13.7798,4.8626,54.5302,NaN,1 (bot)
gap_median_s,219.3301,99.9737,-119.3563,-54.4186,NaN,0 (human)
cookie_age_days,73.9084,48.7264,-25.1820,-34.0719,NaN,0 (human)
gap_cv,2.0356,2.6712,0.6356,31.2256,NaN,1 (bot)
query_nunique,3.7898,4.7586,0.9688,25.5627,NaN,1 (bot)
pointer_present_share,0.3536,0.2653,-0.0883,-24.9761,-8.8307,0 (human)


### Интерпретация добавленных признаков по средним значениям классов

Положительное значение `class_1_minus_class_0` означает, что среднее выше у ботов, отрицательное - у людей.

У ботов больше событий при похожей длительности активности и числе активных часов. Вместе с меньшими интервалами `gap_*` и более сильными минутными всплесками это указывает на более плотную и быструю активность.

Боты также взаимодействуют с большим числом уникальных товаров и запросов. Часть различия объясняется общим количеством событий, поэтому дополнительно используются отношения `nunique / total_events`, доля самого частого значения и entropy.

Маркеры автоматизации в User-Agent чаще встречаются у ботов, а координаты указателя присутствуют реже. Число разных User-Agent и доля точных дубликатов различаются слабее, поэтому это дополнительные, а не основные признаки.

Cookie ботов в среднем моложе. `captcha_count` внутри разрешённого окна не даёт полезного сигнала, а события после `window_end_ts` использовать нельзя из-за утечки.

Наиболее перспективными выглядят количество и скорость событий, burst-активность, разнообразие объектов, возраст cookie и признаки автоматизации.

In [11]:
# Сравниваем типы событий внутри и вне разрешённых окон.
print('event_name by split and window membership:')
display(pd.crosstab([evm['split'], evm['in_window']], evm['event_name'], margins=True))

# Выделяем запрещённые для признаков события train вне окна наблюдения.
outside_train = evm[(evm['split'] == 'train') & (~evm['in_window'])].merge(
    train[['cookie_id', 'target']], on='cookie_id', how='left', validate='many_to_one'
)
# Агрегируем будущие события и captcha по target.
outside_by_target = outside_train.groupby('target').agg(
    cookies=('cookie_id', 'nunique'),
    events=('cookie_id', 'size'),
    captcha=('event_name', lambda s: s.eq('captcha_shown').sum()),
)
print('future events for train cookies by target — forbidden for features:')
display(outside_by_target)
# Переносим число будущих событий на уровень cookie.
future_cookie = train[['cookie_id', 'target']].copy()
future_cookie['future_event_count'] = future_cookie.cookie_id.map(outside_train.groupby('cookie_id').size()).fillna(0)
future_cookie['future_captcha_count'] = future_cookie.cookie_id.map(
    outside_train.loc[outside_train.event_name.eq('captcha_shown')].groupby('cookie_id').size()
).fillna(0)
# Показываем, насколько будущая информация связана с целевым классом.
for col in ['future_event_count', 'future_captcha_count']:
    has = future_cookie[col].gt(0)
    print(f'{col}>0: cookies={int(has.sum())}, target_share={future_cookie.loc[has, "target"].mean():.4f}, target_share_without={future_cookie.loc[~has, "target"].mean():.4f}')

event_name by split and window membership:


event_name       captcha_shown  contact_chat_open  contact_message_sent  contact_phone_show  favorite_add  item_view  login  photo_swipe  search_results_view  seller_page_view  \
split in_window                                                                                                                                                                   
test  True                   0               1768                   844                3242          5259      33748   1761        10133                28081              4854   
train False               7928                577                   258                1028          1753      12731    606         3433                10604              1861   
      True                   0               3780                  1979                7046         12037      74338   3900        22951                61717             10688   
All                       7928               6125                  3081               11316         19049     120817   6267        36517               100402             17403   

event_name          All  
split in_window          
test  True        89690  
train False       40779  
      True       198436  
All              328905

future events for train cookies by target — forbidden for features:


,cookies,events,captcha
target,,,
0,4588,21206,958
1,692,19573,6970


future_event_count>0: cookies=5280, target_share=0.1311, target_share_without=0.0356
future_captcha_count>0: cookies=572, target_share=0.7710, target_share_without=0.0435


In [12]:
# Группируем интервалы и создаём таблицу временных признаков по cookie.
gap_group = ev_train.groupby('cookie_id')['gap_s']
timing_evidence = train[
    ['cookie_id', 'target']
].set_index('cookie_id')
# Считаем продолжительность активности и статистики gap.
timing_evidence['active_duration_s'] = (
    ev_train.groupby('cookie_id').event_ts.max()
    - ev_train.groupby('cookie_id').event_ts.min()
).dt.total_seconds()
timing_evidence['gap_mean_s'] = gap_group.mean()
timing_evidence['gap_std_s'] = gap_group.std()
timing_evidence['gap_median_s'] = gap_group.median()
timing_evidence['gap_q10_s'] = gap_group.quantile(0.10)
timing_evidence['gap_q25_s'] = gap_group.quantile(0.25)
timing_evidence['gap_cv'] = (
    timing_evidence.gap_std_s
    / timing_evidence.gap_mean_s.replace(0, np.nan)
)

# Считаем активные минуты и интенсивность событий внутри минуты.
minute_counts = ev_train.groupby(
    ['cookie_id', 'minute']
).size()
timing_evidence['active_minutes'] = (
    minute_counts.groupby(level=0).size()
)
timing_evidence['events_per_active_minute_mean'] = (
    minute_counts.groupby(level=0).mean()
)
timing_evidence['events_per_active_minute_std'] = (
    minute_counts.groupby(level=0).std()
)
timing_evidence['max_events_per_minute'] = (
    minute_counts.groupby(level=0).max()
)

# Выбираем временные признаки для сравнения классов.
timing_columns = [
    'active_duration_s',
    'gap_mean_s',
    'gap_std_s',
    'gap_median_s',
    'gap_q10_s',
    'gap_q25_s',
    'gap_cv',
    'active_minutes',
    'events_per_active_minute_mean',
    'events_per_active_minute_std',
    'max_events_per_minute',
]
# Выводим средние и медианные значения отдельно для людей и ботов.
display(
    timing_evidence.groupby('target')[timing_columns]
    .agg(['mean', 'median'])
    .T
)

target                                           0             1
active_duration_s             mean    11621.668662  12082.845384
                              median   8326.500000   9092.000000
gap_mean_s                    mean      839.432893    562.149225
                              median    657.294118    425.407407
gap_std_s                     mean     1642.118535   1326.165772
                              median   1732.919959   1368.461303
gap_median_s                  mean      219.330083     99.973743
                              median     59.000000     25.000000
gap_q10_s                     mean       82.345593     52.448939
                              median     15.100000      8.400000
gap_q25_s                     mean      118.248894     68.486592
                              median     29.000000     14.000000
gap_cv                        mean        2.035584      2.671206
                              median      2.064049      2.615818
active_minutes                mean       11.469093     11.783092
                              median      9.000000      9.000000
events_per_active_minute_mean mean        1.428801      2.561437
                              median      1.333333      2.000000
events_per_active_minute_std  mean        0.585351      1.143713
                              median      0.547723      0.894427
max_events_per_minute         mean        2.533359      4.325918
                              median      2.000000      4.000000

### Интервалы между событиями (`gap`)

Для каждой cookie события сортируются по `event_ts`, после чего вычисляется время между соседними событиями:

```text
gap_i = event_ts_i - event_ts_(i-1)
```

Интервалы измеряются в секундах и рассчитываются только внутри разрешённого окна. Для первого события `gap` равен `NaN`, потому что предыдущего события нет.

Признаки `gap_mean`, `gap_median` и квантили описывают обычную скорость действий. `gap_std`, `gap_cv` и `gap_iqr_over_median` показывают разброс и чередование быстрых серий с паузами. `gap_mode_share`, `gap_nunique_ratio` и доли интервалов, кратных заданному числу секунд, помогают находить периодичность. `gap_zero_share` отмечает события с одинаковым timestamp.

По train видно, что у ботов типичные интервалы короче и минутные всплески сильнее. При этом интервалы у них не обязательно регулярнее: быстрые серии могут чередоваться с длинными паузами. Поэтому используются одновременно признаки скорости, разброса, периодичности и burst-активности.

Признаки `gap_*` рассчитываются по исходной последовательности, а `dedup_gap_*` - после удаления полных дубликатов. Их совместное использование помогает отличить реальную частую активность от коротких интервалов, вызванных повторными строками.

Для cookie с одним событием сохраняется `gap_count = 0`, а неопределимые статистики остаются `NaN` и обрабатываются preprocessing модели.

In [13]:
# Строим матрицу количества событий каждого типа для каждой cookie.
event_count_table = pd.crosstab(
    ev_train.cookie_id,
    ev_train.event_name,
).reindex(train.cookie_id, fill_value=0)
event_target = train.set_index('cookie_id').target.reindex(
    event_count_table.index
)

# Считаем по классам наличие, среднее количество и среднюю долю событий.
presence_by_target = event_count_table.gt(0).groupby(
    event_target
).mean()
count_by_target = event_count_table.groupby(
    event_target
).mean()
share_table = event_count_table.div(
    event_count_table.sum(axis=1).replace(0, np.nan),
    axis=0,
)
share_by_target = share_table.groupby(
    event_target
).mean()

# Объединяем статистики людей и ботов в одну сравнительную таблицу.
event_type_evidence = pd.DataFrame({
    'human_presence': presence_by_target.loc[0],
    'bot_presence': presence_by_target.loc[1],
    'presence_diff_pp': (
        presence_by_target.loc[1]
        - presence_by_target.loc[0]
    ) * 100,
    'human_mean_count': count_by_target.loc[0],
    'bot_mean_count': count_by_target.loc[1],
    'human_mean_share': share_by_target.loc[0],
    'bot_mean_share': share_by_target.loc[1],
})
# Добавляем разницу средних долей и сортируем результат.
event_type_evidence['share_diff_pp'] = (
    event_type_evidence.bot_mean_share
    - event_type_evidence.human_mean_share
) * 100
display(
    event_type_evidence.sort_values(
        'presence_diff_pp',
        key=lambda s: s.abs(),
        ascending=False,
    )
)

,human_presence,bot_presence,presence_diff_pp,human_mean_count,bot_mean_count,human_mean_share,bot_mean_share,share_diff_pp
event_name,,,,,,,,
seller_page_view,0.494800,0.676307,18.150716,0.895310,1.738598,0.052861,0.061046,0.818448
favorite_add,0.547684,0.448276,-9.940860,1.110381,0.800890,0.068735,0.033013,-3.572121
login,0.268838,0.192436,-7.640226,0.362343,0.230256,0.023179,0.009808,-1.337154
item_view,0.930043,0.982202,5.215928,6.158752,12.867631,0.364250,0.424916,6.066588
search_results_view,0.913265,0.958843,4.557785,5.141189,10.364850,0.308472,0.331261,2.278836
contact_message_sent,0.136676,0.173526,3.685032,0.173568,0.233593,0.008091,0.010416,0.232519
contact_phone_show,0.348116,0.384872,3.675591,0.624117,0.761958,0.030725,0.033282,0.255652
contact_chat_open,0.228611,0.254727,2.611680,0.333987,0.418242,0.016645,0.018093,0.144858
photo_swipe,0.730279,0.731924,0.164571,2.069368,2.068966,0.127041,0.078164,-4.887626


**Вывод по типам событий.** Наиболее заметно отличается наличие `seller_page_view`: 67.63% бот-cookie против 49.48% человеческих, разница около 18.15 процентного пункта. `favorite_add` и `login`, наоборот, чаще встречаются у людей. Для некоторых событий наличие почти одинаково, но могут различаться количество и доля. Поэтому для каждого типа создаются `event_*_count` и `event_*_share`; target-зависимые частоты из этой таблицы признаками не являются.

In [14]:
# Вычисляем возраст каждой cookie на начало окна наблюдения.
age_evidence = train[
    ['cookie_id', 'target', 'cookie_created_at', 'window_start_ts']
].copy()
age_evidence['cookie_age_days'] = (
    age_evidence.window_start_ts
    - age_evidence.cookie_created_at
).dt.total_seconds() / 86400
# Сравниваем распределение возраста cookie между классами.
age_summary = age_evidence.groupby('target').cookie_age_days.agg(
    count='size',
    mean='mean',
    median='median',
    q25=lambda s: s.quantile(0.25),
    q75=lambda s: s.quantile(0.75),
    q90=lambda s: s.quantile(0.90),
)
display(age_summary)

# Считаем долю молодых cookie при нескольких возрастных границах.
young_cookie_rates = pd.DataFrame(index=[0, 1])
for days in [1, 7, 30, 90]:
    young_cookie_rates[f'age_le_{days}d'] = (
        age_evidence.cookie_age_days.le(days)
        .groupby(age_evidence.target)
        .mean()
    )
display(young_cookie_rates)

,count,mean,median,q25,q75,q90
target,,,,,,
0,10192,73.908394,61.782367,18.850226,110.44640,163.602903
1,899,48.726411,18.968403,0.943495,83.45294,139.057292


,age_le_1d,age_le_7d,age_le_30d,age_le_90d
0,0.104297,0.215757,0.303277,0.659341
1,0.259177,0.481646,0.533927,0.775306


**Вывод по возрасту cookie.** Медианный возраст cookie на начало окна у ботов составляет около 19 дней, у людей - около 62 дней. Поэтому используются возраст в днях, `log1p` возраста (он сжимает длинный правый хвост распределения и помогает LogisticRegression учитывать, что различия среди молодых cookie важнее одинаковых абсолютных различий среди старых) и пороговые признаки молодого cookie. Эти значения известны к началу окна и не создают утечку; абсолютная календарная дата в модель не передаётся, чтобы не запоминать временной тренд train.

### Выводы EDA и защита от утечки

После анализа данных я сделал несколько основных выводов:

- cookie в train и test уникальны и не пересекаются, при этом test расположен позже train;
- `eid` повторяет информацию из `event_name`, поэтому отдельно его использовать нет смысла;
- часть событий train находится после конца окна наблюдения. Использование таких событий дало бы модели информацию из будущего, поэтому признаки строятся только по условию `window_start_ts <= event_ts < window_end_ts`;
- в events есть точные дубликаты. Я не стал просто удалять их, а добавил их долю как отдельный признак и также посчитал временные характеристики после удаления копий;
- значения `platform` были приведены к одному регистру, а из `user_agent` были извлечены отдельные технические признаки вместо использования всей строки;
- у ботов в целом наблюдается более плотная и быстрая активность, а их cookie чаще оказываются молодыми.

Поэтому в feature engineering я использую количество и доли событий, интервалы между действиями, burst-активность, разнообразие значений, повторы и переходы, признаки User-Agent, координаты pointer и возраст cookie. Порядковый номер даты окна в модель не передаётся.

### Схема валидации

Обычный `StratifiedKFold` здесь не подходит, потому что test относится к более позднему периоду. Я использую три последовательных временных разбиения с расширяющейся обучающей выборкой: модель обучается на всех доступных прошлых датах и проверяется на следующем временном блоке. Перемешивание не применяется.

Cookie между строками не повторяются, поэтому отдельный group split не нужен. На каждом временном разбиении считается только `precision_at_recall`, по которой выбирается лучшая модель.

### 2. Реализация feature engineering

In [15]:
import re
from collections.abc import Iterable, Mapping

EVENT_COLUMNS = [
    'cookie_id',
    'event_ts',
    'eid',
    'event_name',
    'platform',
    'user_agent',
    'item_id',
    'item_category',
    'item_location',
    'seller_type',
    'search_query',
    'search_page',
    'pointer_x',
    'pointer_y',
]

#### `events_in_window`

Функция присоединяет к событиям границы окна каждой cookie и оставляет только строки, удовлетворяющие условию `window_start_ts <= event_ts < window_end_ts`. Проверка уникальности `cookie_id` в метаданных и полуоткрытая правая граница защищают построение признаков от временной утечки.

In [16]:
def events_in_window(events: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    """Оставляет только события, для которых window_start_ts <= event_ts < window_end_ts."""
    bounds = meta[["cookie_id", "window_start_ts", "window_end_ts"]]
    if bounds["cookie_id"].duplicated().any():
        raise ValueError("meta.cookie_id must be unique")
    merged = events.merge(bounds, on="cookie_id", how="inner", validate="many_to_one")
    mask = merged["event_ts"].ge(merged["window_start_ts"]) & merged["event_ts"].lt(
        merged["window_end_ts"]
    )
    return merged.loc[mask, EVENT_COLUMNS].copy()

#### `make_feature_vocab`

Функция создаёт фиксированные отсортированные списки допустимых значений для типов событий, платформ, категорий товаров и типов продавцов. Vocabulary строится только по разрешённым train-событиям и не использует `target`; платформа предварительно нормализуется.

In [17]:
def make_feature_vocab(events: pd.DataFrame) -> dict[str, list[str]]:
    """Создаёт словари значений без target по событиям из разрешённых train-окон."""
    platform = events["platform"].str.lower().replace({"iphone": "ios"})
    return {
        "event_name": sorted(events["event_name"].dropna().astype(str).unique()),
        "platform_norm": sorted(platform.dropna().astype(str).unique()),
        "item_category": sorted(events["item_category"].dropna().astype(str).unique()),
        "seller_type": sorted(events["seller_type"].dropna().astype(str).unique()),
    }

#### `_safe_name`

Служебная функция преобразует произвольное значение категории в безопасную часть имени колонки: заменяет специальные символы на подчёркивания, удаляет крайние подчёркивания и приводит текст к нижнему регистру.

In [18]:
def _safe_name(value: object) -> str:
    """Преобразует значение категории в безопасную часть имени признака."""
    return re.sub(r"[^0-9a-zA-Z_]+", "_", str(value)).strip("_").lower()

#### `_join`

Служебная функция выполняет левое присоединение агрегированного Series или DataFrame к таблице признаков. Благодаря левому join сохраняется полный индекс cookie, даже если для части cookie конкретная статистика отсутствует.

In [19]:
def _join(features: pd.DataFrame, values: pd.Series | pd.DataFrame) -> pd.DataFrame:
    """Присоединяет агрегированные значения, сохраняя все cookie из таблицы признаков."""
    return features.join(values, how="left")

#### `_distribution_stats`

Функция описывает распределение значений одной событийной колонки внутри каждой cookie. Она рассчитывает число заполненных значений, количество уникальных значений, отношение уникальных значений к числу наблюдений, долю моды, entropy и нормализованную entropy, которые потом становятся признаками.

In [20]:
def _distribution_stats(events: pd.DataFrame, column: str, prefix: str) -> pd.DataFrame:
    """Считает разнообразие, частоту лидирующего значения и entropy по каждой cookie."""
    nonnull = events.loc[events[column].notna(), ["cookie_id", column]]
    if nonnull.empty:
        return pd.DataFrame()

# Считаем частоты значений и вероятности внутри каждой cookie.
    counts = nonnull.groupby(["cookie_id", column], observed=True).size().rename("count")
    totals = counts.groupby(level=0).sum()
    probabilities = counts / counts.groupby(level=0).transform("sum")
    entropy = (-(probabilities * np.log(probabilities))).groupby(level=0).sum()
    nunique = counts.groupby(level=0).size()
    top_share = counts.groupby(level=0).max() / totals
    normalized_entropy = entropy / np.log(nunique.where(nunique.gt(1)))

# Собираем число значений, разнообразие, top-share и entropy.
    out = pd.DataFrame(index=totals.index)
    out[f"{prefix}_nonnull_count"] = totals
    out[f"{prefix}_nunique"] = nunique
    out[f"{prefix}_nunique_ratio"] = nunique / totals
    out[f"{prefix}_top_share"] = top_share
    out[f"{prefix}_entropy"] = entropy
    out[f"{prefix}_entropy_norm"] = normalized_entropy.fillna(0.0)
    return out

#### `_count_share_features`

Функция для каждого значения из заранее построенного vocabulary создаёт два признака: абсолютное количество событий и долю этих событий среди всей активности cookie. Фиксированный vocabulary гарантирует одинаковый набор колонок в train и test.

In [21]:
def _count_share_features(
    events: pd.DataFrame,
    column: str,
    values: Iterable[str],
    prefix: str,
) -> pd.DataFrame:
    """Создаёт количество и долю событий для каждого значения категории."""
    table = events.groupby(["cookie_id", column], observed=True).size().unstack(fill_value=0)
    total = events.groupby("cookie_id").size()
# Для каждого значения создаём абсолютное количество и долю.
    out = pd.DataFrame(index=table.index)
    for value in values:
        count = table[value] if value in table.columns else pd.Series(0, index=table.index)
        name = _safe_name(value)
        out[f"{prefix}_{name}_count"] = count
        out[f"{prefix}_{name}_share"] = count / total
    return out

#### `_gap_features`

Это техническая реализация признаков, подробно разобранных в разделе временного EDA выше. Функция принимает временную последовательность и `prefix`, поэтому один и тот же расчёт применяется дважды: с префиксом `gap` для исходных событий и `dedup_gap` для последовательности без полных копий. На выходе получается одна строка агрегатов на `cookie_id`.

In [22]:
def _gap_features(sequence: pd.DataFrame, prefix: str) -> pd.DataFrame:
    """Строит статистики интервалов между последовательными событиями каждой cookie."""
    sequence = sequence.sort_values(["cookie_id", "event_ts"], kind="mergesort").copy()
    sequence["gap_s"] = sequence.groupby("cookie_id", sort=False)["event_ts"].diff().dt.total_seconds()
    gap_group = sequence.groupby("cookie_id", sort=False)["gap_s"]
# Считаем основные агрегаты распределения gap.
    out = gap_group.agg(["count", "mean", "std", "min", "max", "median"])
    out.columns = [f"{prefix}_{name}" for name in out.columns]

# Добавляем квантили и относительные меры разброса.
    for quantile in (0.10, 0.25, 0.75, 0.90, 0.95):
        out[f"{prefix}_q{int(quantile * 100):02d}"] = gap_group.quantile(quantile)
    out[f"{prefix}_cv"] = out[f"{prefix}_std"] / out[f"{prefix}_mean"].replace(0, np.nan)
    out[f"{prefix}_iqr_over_median"] = (
        out[f"{prefix}_q75"] - out[f"{prefix}_q25"]
    ) / out[f"{prefix}_median"].replace(0, np.nan)

# Оцениваем повторяемость и регулярность ненулевых интервалов.
    valid = sequence.loc[sequence["gap_s"].notna(), ["cookie_id", "gap_s"]].copy()
    if not valid.empty:
        by_gap = valid.groupby(["cookie_id", "gap_s"], observed=True).size()
        interval_count = by_gap.groupby(level=0).sum()
        gap_nunique = by_gap.groupby(level=0).size()
        out[f"{prefix}_mode_share"] = by_gap.groupby(level=0).max() / interval_count
        out[f"{prefix}_nunique_ratio"] = gap_nunique / interval_count
        for seconds in (5, 10, 30, 60):
            share = valid["gap_s"].mod(seconds).eq(0).groupby(valid["cookie_id"]).mean()
            out[f"{prefix}_multiple_{seconds}s_share"] = share
        out[f"{prefix}_zero_share"] = valid["gap_s"].eq(0).groupby(valid["cookie_id"]).mean()
    return out

#### `_burst_features`

Функция измеряет кратковременные всплески активности. Для каждой cookie алгоритм скользящего временного окна находит максимальное число событий, произошедших за 10, 60 и 300 секунд.

In [23]:
def _burst_features(sequence: pd.DataFrame) -> pd.DataFrame:
    """Находит максимальное число событий в коротких временных интервалах."""
    rows: list[dict[str, float | str]] = []
# Просматриваем временную последовательность каждой cookie.
    for cookie_id, part in sequence.groupby("cookie_id", sort=False):
        times = part["event_ts"].astype("int64").to_numpy() / 1_000_000.0
        row: dict[str, float | str] = {"cookie_id": cookie_id}
        starts = np.arange(len(times))
# Считаем максимум событий в окнах 10, 60 и 300 секунд.
        for seconds in (10, 60, 300):
            ends = np.searchsorted(times, times + seconds, side="left")
            row[f"max_events_{seconds}s"] = float((ends - starts).max())
        rows.append(row)
# Возвращаем одну строку burst-признаков на cookie.
    return pd.DataFrame(rows).set_index("cookie_id")

#### `build_cookie_features`

Главная функция feature engineering создаёт одну строку числовых признаков на cookie. Она повторно фильтрует события по окну, добавляет метаданные cookie, counts/shares, разнообразие категорий, признаки User-Agent и дубликатов, временные интервалы, burst-активность, последовательности, сессии, search page и pointer-статистики. Порядок строк сверяется с исходными метаданными.

In [24]:
def build_cookie_features(
    events: pd.DataFrame,
    meta: pd.DataFrame,
    vocab: Mapping[str, Iterable[str]],
) -> pd.DataFrame:
    """Строит числовые признаки только из окна наблюдения каждой cookie."""
    ev = events_in_window(events, meta)
    features = meta[["cookie_id"]].drop_duplicates().set_index("cookie_id")

# Строим возрастные и календарные признаки из метаданных cookie.
    metadata = meta.set_index("cookie_id")
    age_start = (metadata["window_start_ts"] - metadata["cookie_created_at"]).dt.total_seconds() / 86400
    age_end = (metadata["window_end_ts"] - metadata["cookie_created_at"]).dt.total_seconds() / 86400
    features["cookie_age_start_days"] = age_start
    features["cookie_age_end_days"] = age_end
    features["cookie_age_log1p"] = np.log1p(age_start.clip(lower=0))
    features["cookie_created_hour"] = metadata["cookie_created_at"].dt.hour
    features["cookie_created_dow"] = metadata["cookie_created_at"].dt.dayofweek
    features["window_dow"] = metadata["window_start_ts"].dt.dayofweek
    for days in (1, 7, 30, 90):
        features[f"cookie_age_le_{days}d"] = age_start.le(days).astype(int)

# Нормализуем platform и извлекаем устойчивые признаки из User-Agent.
    ev["platform_norm"] = ev["platform"].str.lower().replace({"iphone": "ios"})
    ua = ev["user_agent"].fillna("").str.lower()
    ev["ua_headless"] = ua.str.contains("headless", regex=False)
    ev["ua_automation"] = ua.str.contains(
        "bot|python|requests|curl|wget|scrapy|selenium|playwright", regex=True
    )
    ev["ua_mobile"] = ua.str.contains("mobile|android|iphone|ipad", regex=True)
    ev["ua_linux"] = ua.str.contains("linux", regex=False)
    ev["ua_windows"] = ua.str.contains("windows", regex=False)
    ev["ua_macos"] = ua.str.contains("macintosh|mac os", regex=True)

# Считаем общий объём и число типов событий.
    total = ev.groupby("cookie_id", sort=False).size().rename("total_events")
    features = _join(features, total)
    features = _join(features, ev.groupby("cookie_id")["event_name"].nunique().rename("event_type_nunique"))

# Создаём counts и shares для событий и низкокардинальных категорий.
    event_features = _count_share_features(
        ev, "event_name", vocab["event_name"], "event"
    )
    platform_features = _count_share_features(
        ev, "platform_norm", vocab["platform_norm"], "platform"
    )
    category_features = _count_share_features(
        ev, "item_category", vocab["item_category"], "category"
    )
    seller_features = _count_share_features(
        ev, "seller_type", vocab["seller_type"], "seller"
    )
    for block in (event_features, platform_features, category_features, seller_features):
        features = _join(features, block)

# Считаем разнообразие, top-share и entropy категориальных значений.
    distribution_columns = [
        "event_name",
        "platform_norm",
        "user_agent",
        "item_id",
        "item_category",
        "item_location",
        "seller_type",
        "search_query",
        "search_page",
    ]
    for column in distribution_columns:
        features = _join(
            features,
            _distribution_stats(ev, column, f"{_safe_name(column)}_dist"),
        )

# Агрегируем бинарные признаки User-Agent до уровня cookie.
    for flag in (
        "ua_headless",
        "ua_automation",
        "ua_mobile",
        "ua_linux",
        "ua_windows",
        "ua_macos",
    ):
        features = _join(features, ev.groupby("cookie_id")[flag].mean().rename(f"{flag}_share"))

# Измеряем точные дубликаты и число событий после их удаления.
    original_columns = [column for column in EVENT_COLUMNS if column in ev.columns]
    ev["is_exact_duplicate"] = ev.duplicated(subset=original_columns, keep=False)
    features = _join(
        features,
        ev.groupby("cookie_id")["is_exact_duplicate"].agg(
            duplicate_row_count="sum", duplicate_row_share="mean"
        ),
    )
    deduplicated = ev.drop_duplicates(subset=original_columns, keep="first")
    features = _join(
        features,
        deduplicated.groupby("cookie_id").size().rename("deduplicated_events"),
    )

# Строим границы и продолжительность наблюдаемой активности.
    sequence = ev.sort_values(["cookie_id", "event_ts"], kind="mergesort").copy()
    time_group = sequence.groupby("cookie_id", sort=False)["event_ts"]
    first_event = time_group.min()
    last_event = time_group.max()
    features = _join(features, (last_event - first_event).dt.total_seconds().rename("active_duration_s"))
    features = _join(
        features,
        (first_event - metadata["window_start_ts"]).dt.total_seconds().rename("start_to_first_s"),
    )
    features = _join(
        features,
        (metadata["window_end_ts"] - last_event).dt.total_seconds().rename("last_to_end_s"),
    )
# Добавляем gap-признаки для исходной и очищенной последовательностей.
    features = _join(features, _gap_features(sequence, "gap"))
    features = _join(features, _gap_features(deduplicated, "dedup_gap"))

# Считаем активность по часам.
    sequence["event_hour"] = sequence["event_ts"].dt.hour
    sequence["event_minute"] = sequence["event_ts"].dt.floor("min")
    features = _join(
        features,
        sequence.groupby("cookie_id")["event_hour"].nunique().rename("active_hours"),
    )
    features = _join(features, _distribution_stats(sequence, "event_hour", "hour"))

# Измеряем минутную интенсивность и короткие burst-периоды.
    minute_counts = sequence.groupby(["cookie_id", "event_minute"]).size()
    minute_stats = minute_counts.groupby(level=0).agg(["size", "mean", "std", "max"])
    minute_stats.columns = ["active_minutes", "events_per_active_minute_mean", "events_per_active_minute_std", "max_events_per_minute"]
    minute_stats["top_minute_event_share"] = minute_counts.groupby(level=0).max() / total
    minute_stats["burst_minutes_ge_3"] = minute_counts.ge(3).groupby(level=0).sum()
    minute_stats["burst_minutes_ge_5"] = minute_counts.ge(5).groupby(level=0).sum()
    features = _join(features, minute_stats)
    features = _join(features, _burst_features(sequence))

# Строим признаки соседних повторов и переходов между событиями.
    previous_event = sequence.groupby("cookie_id", sort=False)["event_name"].shift()
    has_previous = previous_event.notna()
    sequence["same_as_previous"] = sequence["event_name"].eq(previous_event) & has_previous
    features = _join(
        features,
        sequence.loc[has_previous].groupby("cookie_id")["same_as_previous"].mean().rename("repeat_adjacent_share"),
    )
    transition_frame = sequence.loc[has_previous, ["cookie_id", "event_name"]].copy()
    transition_frame["transition"] = previous_event.loc[has_previous] + "__to__" + transition_frame["event_name"]
    features = _join(features, _distribution_stats(transition_frame, "transition", "transition"))

# Считаем количество и длину серий одинаковых действий.
    run_start = previous_event.isna() | sequence["event_name"].ne(previous_event)
    sequence["run_id"] = run_start.groupby(sequence["cookie_id"]).cumsum()
    run_sizes = sequence.groupby(["cookie_id", "run_id"]).size()
    run_stats = run_sizes.groupby(level=0).agg(["size", "mean", "max"])
    run_stats.columns = ["same_action_run_count", "same_action_run_mean", "same_action_run_max"]
    features = _join(features, run_stats)

# Разделяем активность на сессии по паузе более 30 минут.
    gap_s = sequence.groupby("cookie_id", sort=False)["event_ts"].diff().dt.total_seconds()
    sequence["session_id"] = gap_s.gt(1800).groupby(sequence["cookie_id"]).cumsum()
    session_sizes = sequence.groupby(["cookie_id", "session_id"]).size()
    session_stats = session_sizes.groupby(level=0).agg(["size", "mean", "max"])
    session_stats.columns = ["session_count", "session_events_mean", "session_events_max"]
    features = _join(features, session_stats)

# Агрегируем числовые поля search_page и координаты pointer.
    numeric_event_columns = ["search_page", "pointer_x", "pointer_y"]
    for column in numeric_event_columns:
        block = ev.groupby("cookie_id")[column].agg(["count", "mean", "std", "min", "max", "median", "nunique"])
        block.columns = [f"{column}_{name}" for name in block.columns]
        block[f"{column}_present_share"] = block[f"{column}_count"] / total
        features = _join(features, block)

# Очищаем бесконечные значения и проверяем порядок cookie.
    features = features.replace([np.inf, -np.inf], np.nan).reset_index()
    if not features["cookie_id"].equals(meta["cookie_id"].reset_index(drop=True)):
        raise AssertionError("Feature rows no longer match metadata order")
    return features

#### `build_additional_features`

Функция отдельно строит дополнительные признаки движения pointer, переходов между страницами поиска и повторов соседних item/category. Они не смешиваются с основным набором, чтобы их пользу можно было проверить на той же временной валидации.

In [25]:
def build_additional_features(events: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    """Строит дополнительные признаки pointer и переходов между страницами отдельно от основного набора."""
    sequence = events_in_window(events, meta).sort_values(
        ["cookie_id", "event_ts"], kind="mergesort"
    )
    features = meta[["cookie_id"]].drop_duplicates().set_index("cookie_id")

# Выбираем события с одновременно заполненными координатами pointer.
    pointer_events = sequence.loc[
        sequence["pointer_x"].notna() & sequence["pointer_y"].notna(),
        ["cookie_id", "event_ts", "pointer_x", "pointer_y"],
    ].copy()
# Считаем смещения, расстояния и скорости движения pointer.
    if not pointer_events.empty:
        pointer_group = pointer_events.groupby("cookie_id", sort=False)
        pointer_events["pointer_dx"] = pointer_group["pointer_x"].diff()
        pointer_events["pointer_dy"] = pointer_group["pointer_y"].diff()
        pointer_events["pointer_gap_s"] = pointer_group["event_ts"].diff().dt.total_seconds()
        pointer_events["pointer_distance"] = np.hypot(
            pointer_events["pointer_dx"], pointer_events["pointer_dy"]
        )
        pointer_events["pointer_speed"] = pointer_events["pointer_distance"] / pointer_events[
            "pointer_gap_s"
        ].where(pointer_events["pointer_gap_s"].gt(0))
# Агрегируем характеристики движения до уровня cookie.
        pointer_motion = pointer_events.groupby("cookie_id").agg(
            pointer_distance_mean=("pointer_distance", "mean"),
            pointer_distance_median=("pointer_distance", "median"),
            pointer_distance_std=("pointer_distance", "std"),
            pointer_distance_max=("pointer_distance", "max"),
            pointer_speed_mean=("pointer_speed", "mean"),
            pointer_speed_max=("pointer_speed", "max"),
            pointer_x_range=("pointer_x", lambda s: s.max() - s.min()),
            pointer_y_range=("pointer_y", lambda s: s.max() - s.min()),
        )
        pointer_motion["pointer_bbox_area"] = (
            pointer_motion["pointer_x_range"] * pointer_motion["pointer_y_range"]
        )
# Добавляем долю соседних событий без перемещения pointer.
        valid_pointer_move = pointer_events["pointer_distance"].notna()
        pointer_motion["pointer_zero_distance_share"] = pointer_events.loc[
            valid_pointer_move, "pointer_distance"
        ].eq(0).groupby(pointer_events.loc[valid_pointer_move, "cookie_id"]).mean()
        features = _join(features, pointer_motion)

# Считаем долю повторов соседних item и category.
    for column in ("item_id", "item_category"):
        previous = sequence.groupby("cookie_id", sort=False)[column].shift()
        valid_pair = sequence[column].notna() & previous.notna()
        same_share = sequence.loc[valid_pair, column].eq(previous.loc[valid_pair]).groupby(
            sequence.loc[valid_pair, "cookie_id"]
        ).mean()
        features = _join(features, same_share.rename(f"same_{column}_adjacent_share"))

# Анализируем направление и размер переходов между страницами поиска.
    page_events = sequence.loc[
        sequence["search_page"].notna(), ["cookie_id", "search_page"]
    ].copy()
    if not page_events.empty:
        page_events["page_diff"] = page_events.groupby("cookie_id")["search_page"].diff()
        page_group = page_events.groupby("cookie_id")["page_diff"]
        page_motion = page_group.agg(["mean", "std", "min", "max", "median"])
        page_motion.columns = [f"search_page_diff_{name}" for name in page_motion.columns]
        valid_page_diff = page_events["page_diff"].notna()
        page_motion["search_page_same_adjacent_share"] = page_events.loc[
            valid_page_diff, "page_diff"
        ].eq(0).groupby(page_events.loc[valid_page_diff, "cookie_id"]).mean()
        page_motion["search_page_forward_share"] = page_events.loc[
            valid_page_diff, "page_diff"
        ].gt(0).groupby(page_events.loc[valid_page_diff, "cookie_id"]).mean()
        features = _join(features, page_motion)

# Заменяем бесконечные значения на пропуски и возвращаем результат.
    return features.replace([np.inf, -np.inf], np.nan).reset_index()

In [26]:
# Строим vocabulary только по событиям из разрешённых train-окон.
train_window_events = events_in_window(events, train)
feature_vocab = make_feature_vocab(train_window_events)
print({name: len(values) for name, values in feature_vocab.items()})

# Создаём полный набор признаков для train.
train_features = build_cookie_features(events, train, feature_vocab)
feature_columns = [
    column
    for column in train_features.columns
    if column != 'cookie_id'
]

assert train_features.cookie_id.equals(train.cookie_id)
assert not train_features.cookie_id.duplicated().any()

print('train feature shape:', train_features.shape)
print(
    'features with any missing values:',
    int(train_features[feature_columns].isna().any().sum()),
)
display(train_features.head())

{'event_name': 9, 'platform_norm': 4, 'item_category': 15, 'seller_type': 2}
train feature shape: (11091, 233)
features with any missing values: 131


,cookie_id,cookie_age_start_days,cookie_age_end_days,cookie_age_log1p,cookie_created_hour,cookie_created_dow,window_dow,cookie_age_le_1d,cookie_age_le_7d,cookie_age_le_30d,cookie_age_le_90d,total_events,event_type_nunique,event_contact_chat_open_count,event_contact_chat_open_share,event_contact_message_sent_count,event_contact_message_sent_share,event_contact_phone_show_count,event_contact_phone_show_share,event_favorite_add_count,event_favorite_add_share,event_item_view_count,event_item_view_share,event_login_count,event_login_share,event_photo_swipe_count,event_photo_swipe_share,event_search_results_view_count,event_search_results_view_share,event_seller_page_view_count,event_seller_page_view_share,platform_android_count,platform_android_share,platform_desktop_count,platform_desktop_share,platform_ios_count,platform_ios_share,platform_web_count,platform_web_share,category_avtomobili_count,category_avtomobili_share,category_bytovaya_tehnika_count,category_bytovaya_tehnika_share,category_detskie_tovary_count,category_detskie_tovary_share,category_elektronika_count,category_elektronika_share,category_hobbi_count,category_hobbi_share,category_kvartiry_arenda_count,...,hour_top_share,hour_entropy,hour_entropy_norm,active_minutes,events_per_active_minute_mean,events_per_active_minute_std,max_events_per_minute,top_minute_event_share,burst_minutes_ge_3,burst_minutes_ge_5,max_events_10s,max_events_60s,max_events_300s,repeat_adjacent_share,transition_nonnull_count,transition_nunique,transition_nunique_ratio,transition_top_share,transition_entropy,transition_entropy_norm,same_action_run_count,same_action_run_mean,same_action_run_max,session_count,session_events_mean,session_events_max,search_page_count,search_page_mean,search_page_std,search_page_min,search_page_max,search_page_median,search_page_nunique,search_page_present_share,pointer_x_count,pointer_x_mean,pointer_x_std,pointer_x_min,pointer_x_max,pointer_x_median,pointer_x_nunique,pointer_x_present_share,pointer_y_count,pointer_y_mean,pointer_y_std,pointer_y_min,pointer_y_max,pointer_y_median,pointer_y_nunique,pointer_y_present_share
0,ck_54a059eb7d3ea68b,135.603692,136.603692,4.917084,9,4,0,0,0,0,0,7,3,0,0.000000,0,0.000000,0,0.000000,0,0.000000,3,0.428571,0,0.000000,1,0.142857,3,0.428571,0,0.000000,7,1.0,0,0.000000,0,0.0,0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,1.0,0.142857,0.0,0.000000,0.0,...,1.000000,0.000000,0.000000,4,1.750000,0.500000,2,0.285714,0,0,2.0,4.0,7.0,0.333333,6.0,5.0,0.833333,0.333333,1.560710,0.969724,5,1.400000,2,1,7.000000,7,3,3.000000,2.645751,1.0,6.0,2.0,3,0.428571,0,NaN,NaN,NaN,NaN,NaN,0,0.000000,0,NaN,NaN,NaN,NaN,NaN,0,0.000000
1,ck_7e4de46eeab82974,194.576111,195.576111,5.275950,10,1,0,0,0,0,0,41,7,0,0.000000,0,0.000000,1,0.024390,1,0.024390,22,0.536585,1,0.024390,4,0.097561,8,0.195122,4,0.097561,0,0.0,10,0.243902,0,0.0,31,0.756098,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,39.0,...,0.292683,1.857061,0.893058,30,1.366667,0.668675,3,0.073171,3,0,2.0,4.0,9.0,0.425000,40.0,18.0,0.450000,0.375000,2.335282,0.807952,24,1.708333,8,7,5.857143,13,8,3.625000,2.615203,1.0,8.0,3.5,6,0.195122,38,917.868421,509.231969,2.0,1907.0,845.0,36,0.926829,38,567.605263,272.953569,104.0,1053.0,601.0,38,0.926829
2,ck_9320229ef6304522,32.994421,33.994421,3.526196,0,2,0,0,0,0,1,36,9,1,0.027778,1,0.027778,2,0.055556,3,0.083333,7,0.194444,3,0.083333,3,0.083333,13,0.361111,3,0.083333,0,0.0,8,0.222222,0,0.0,28,0.777778,2.0,0.055556,6.0,0.166667,8.0,0.222222,0.0,0.000000,3.0,0.083333,0.0,...,0.277778,1.843821,0.947537,21,1.714286,0.902378,4,0.111111,4,0,2.0,5.0,7.0,0.142857,35.0,23.0,0.657143,0.085714,3.020247,0.963244,31,1.161290,3,7,5.142857,10,13,3.538462,3.204564,1.0,10.0,2.0,7,0.361111,35,936.457143,602.124295,35.0,1904.0,989.0,35,0.972222,35,556.657143,312.027426,22.0,1057.0,560.0,35,0.972222
3,ck_30ccd25bc1714ed9,0.546759,1.546759,0.436162,10,6,0,1,1,1,1,21,5,1,0.047619,0,0.000000,1,0.047619,0,0.000000,14,0.666667,0,0.000000,0,0.000000,3,0.142857,2

## 3. Два варианта baseline: LogisticRegression

Сначала обучается простой baseline на 27 понятных признаках: возраст cookie, количество и доли типов событий, а также базовые показатели разнообразия. Затем та же `LogisticRegression` обучается на всех 232 признаках после feature engineering. Модель, preprocessing и схема временной валидации одинаковые, поэтому разница в метриках показывает вклад расширенного набора признаков.

In [27]:
# Импортируем функцию клонирования моделей.
from sklearn.base import clone

# Задаём последовательные интервалы временной валидации.
fold_specs = [
    ('fold_1', '2026-04-11', '2026-04-14'),
    ('fold_2', '2026-04-14', '2026-04-17'),
    ('fold_3', '2026-04-17', '2026-04-20'),
]

def make_temporal_splits(meta, specs=fold_specs):
    """Формирует расширяющиеся временные разбиения без перемешивания строк."""
    splits = []
    for name, valid_start, valid_end in specs:
        # Все более ранние даты включаем в расширяющуюся обучающую часть.
        train_idx = np.flatnonzero(meta.window_start_ts.lt(valid_start).to_numpy())
        valid_idx = np.flatnonzero(
            meta.window_start_ts.ge(valid_start).to_numpy()
            & meta.window_start_ts.lt(valid_end).to_numpy()
        )
        # Проверяем отсутствие пересечения и правильный порядок времени.
        assert len(set(train_idx).intersection(valid_idx)) == 0
        assert meta.iloc[train_idx].window_start_ts.max() < meta.iloc[valid_idx].window_start_ts.min()
        splits.append((name, train_idx, valid_idx))
    return splits

def best_threshold_at_recall(y_true, scores, min_recall=TARGET_RECALL):
    """Возвращает порог с максимальной precision при заданном минимальном recall."""
    # Сортируем объекты по score и сохраняем одинаковые score одной группой.
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)
    positive_count = int(y_true.sum())
    if positive_count == 0:
        return float('nan')

    order = np.argsort(-scores, kind='mergesort')
    sorted_target = y_true[order]
    sorted_scores = scores[order]
    group_ends = np.r_[sorted_scores[1:] != sorted_scores[:-1], True]

    # Считаем precision и recall только на границах групп одинакового score.
    true_positive = np.cumsum(sorted_target)[group_ends]
    predicted_positive = np.arange(1, len(sorted_target) + 1)[group_ends]
    precision = true_positive / predicted_positive
    recall = true_positive / positive_count
    thresholds = sorted_scores[group_ends]

    # Среди порогов с достаточным recall выбираем максимальную precision.
    valid_indices = np.flatnonzero(recall >= min_recall)
    if len(valid_indices) == 0:
        return float('nan')
    best_index = valid_indices[np.argmax(precision[valid_indices])]
    return float(thresholds[best_index])
def cross_validate_scores(estimator, X, y, splits):
    """Обучает модель на временных разбиениях и возвращает Precision@Recall, OOF-прогнозы и модели."""
    oof = np.full(len(y), np.nan)
    rows = []
    fitted = []
    for name, train_idx, valid_idx in splits:
        model = clone(estimator)
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        scores = model.predict_proba(X.iloc[valid_idx])[:, 1]
        oof[valid_idx] = scores
        # Сохраняем Precision@Recall и порог validation-блока.
        rows.append({
            'fold': name,
            'train_size': len(train_idx),
            'valid_size': len(valid_idx),
            'valid_positives': int(y.iloc[valid_idx].sum()),
            'precision_at_recall': precision_at_recall(y.iloc[valid_idx], scores),
            'threshold': best_threshold_at_recall(y.iloc[valid_idx], scores),
        })
        fitted.append(model)
    return pd.DataFrame(rows), oof, fitted

# Создаём временные разбиения и выводим размеры классов.
temporal_splits = make_temporal_splits(train)
for name, train_idx, valid_idx in temporal_splits:
    print(name, 'train=', len(train_idx), 'valid=', len(valid_idx),
          'train_pos=', int(train.target.iloc[train_idx].sum()),
          'valid_pos=', int(train.target.iloc[valid_idx].sum()))

fold_1 train= 4217 valid= 2556 train_pos= 345 valid_pos= 199
fold_2 train= 6773 valid= 2367 train_pos= 544 valid_pos= 195
fold_3 train= 9140 valid= 1951 train_pos= 739 valid_pos= 160


In [28]:
# Импортируем обработку пропусков, масштабирование и LogisticRegression.
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Выбираем простой набор counts, shares и признаков разнообразия.
event_basic = [
    c for c in feature_columns
    if c.startswith('event_')
    and not c.startswith('event_name_dist_')
    and (c.endswith('_count') or c.endswith('_share'))
]
diversity_basic = [
    'item_id_dist_nunique', 'item_category_dist_nunique', 'item_location_dist_nunique',
    'search_query_dist_nunique', 'platform_norm_dist_nunique', 'user_agent_dist_nunique',
]
# Формируем 27 колонок компактного baseline.
baseline_columns = [
    'cookie_age_log1p',
    'total_events',
    'event_type_nunique',
] + event_basic + diversity_basic

# Собираем pipeline: заполнение NaN, StandardScaler и LogisticRegression.
baseline_model = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=0.0)),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        C=1.0,
        class_weight='balanced',
        max_iter=2000,
        random_state=RANDOM_STATE,
    )),
])

# Проверяем baseline на 27 признаках по временным разбиениям.
baseline_results, baseline_oof, baseline_models = cross_validate_scores(
    baseline_model,
    train_features[baseline_columns],
    train['target'],
    temporal_splits,
)

# Повторяем ту же проверку на полном наборе признаков.
full_lr_results, full_lr_oof, full_lr_models = cross_validate_scores(
    baseline_model,
    train_features[feature_columns],
    train['target'],
    temporal_splits,
)

# Объединяем результаты двух вариантов по каждому временному разбиению.
baseline_fold_comparison = pd.concat([
    baseline_results.assign(model=f'LogisticRegression: {len(baseline_columns)} признаков'),
    full_lr_results.assign(model=f'LogisticRegression: все {len(feature_columns)} признака'),
], ignore_index=True)

print('Результаты двух вариантов по каждому временному разбиению:')
display(
    baseline_fold_comparison[
        ['model', 'fold', 'precision_at_recall']
    ]
)

# Считаем среднее значение Precision@Recall и разброс между временными разбиениями.
baseline_summary = baseline_fold_comparison.groupby('model').agg(
    precision_mean=('precision_at_recall', 'mean'),
    precision_std=('precision_at_recall', 'std'),
).sort_values('precision_mean', ascending=False)

print('Средний Precision@Recall по временным разбиениям:')
display(baseline_summary)

Результаты двух вариантов по каждому временному разбиению:


,model,fold,precision_at_recall
0,LogisticRegression: 27 признаков,fold_1,0.243151
1,LogisticRegression: 27 признаков,fold_2,0.311659
2,LogisticRegression: 27 признаков,fold_3,0.287179
3,LogisticRegression: все 232 признака,fold_1,0.343980
4,LogisticRegression: все 232 признака,fold_2,0.556911
5,LogisticRegression: все 232 признака,fold_3,0.511416


Средний Precision@Recall по временным разбиениям:


,precision_mean,precision_std
model,,
LogisticRegression: все 232 признака,0.470769,0.112134
LogisticRegression: 27 признаков,0.280663,0.034716


### Вывод по двум вариантам baseline

Сначала я обучил `LogisticRegression` на 27 простых признаках. Средний `Precision@Recall≥0.70` получился **0.2807**. После этого я оставил ту же модель и ту же схему временной валидации, но передал все 232 признака после feature engineering. Метрика выросла до **0.4708**.

Прирост составил примерно 0.19 по Precision, то есть около 68% относительно простого варианта. Улучшение произошло на каждом временном разбиении:

- первое разбиение: 0.2432 -> 0.3440;
- второе разбиение: 0.3117 -> 0.5569;
- третье разбиение: 0.2872 -> 0.5114.

Дополнительные признаки действительно содержат полезную информацию. Больше всего могли помочь интервалы `gap`, burst-активность, возраст cookie, entropy, признаки User-Agent, повторов и последовательностей событий.

## 4. Переход к более сложным моделям

Сравнение baseline подтвердило, что feature engineering улучшает качество. Однако `LogisticRegression` строит линейную границу, тогда как признаки активности могут быть полезны только при определённых порогах и в сочетаниях. Поэтому дальше сравниваются модели на деревьях, способные находить нелинейные зависимости.

In [29]:
# Импортируем таймер и реализации моделей для табличных данных.
import time

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from xgboost import XGBClassifier

# Передаём всем кандидатам одинаковый набор основных признаков.
all_model_columns = feature_columns
# Задаём стартовые конфигурации моделей без подбора параметров.
candidate_models = {
    'lightgbm': LGBMClassifier(
        objective='binary',
        n_estimators=450,
        learning_rate=0.03,
        num_leaves=15,
        max_depth=-1,
        min_child_samples=40,
        subsample=0.9,
        colsample_bytree=0.85,
        reg_alpha=0.2,
        reg_lambda=2.0,
        scale_pos_weight=3.0,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        deterministic=True,
        force_col_wise=True,
        verbosity=-1,
    ),
    'catboost': CatBoostClassifier(
        iterations=450,
        depth=6,
        learning_rate=0.04,
        l2_leaf_reg=5.0,
        random_seed=RANDOM_STATE,
        auto_class_weights='Balanced',
        loss_function='Logloss',
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
    ),
# При количестве событий больше 10000 рекомендуется использовать
# HistGradientBoosting вместо обычно GradientBoosting
    'hist_gradient_boosting': HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=300,
        max_leaf_nodes=15,
        min_samples_leaf=30,
        l2_regularization=2.0,
        class_weight='balanced',
        random_state=RANDOM_STATE,
    ),
    'xgboost': XGBClassifier(
        objective='binary:logistic',
        n_estimators=450,
        learning_rate=0.03,
        max_depth=5,
        min_child_weight=5.0,
        subsample=0.9,
        colsample_bytree=0.85,
        reg_alpha=0.2,
        reg_lambda=2.0,
        scale_pos_weight=3.0,
        tree_method='hist',
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0,
    ),
}

candidate_results = {}
candidate_oof = {}
candidate_fitted = {}
# Проверяем каждого кандидата на одинаковых временных разбиениях.
for model_name, estimator in candidate_models.items():
    started = time.perf_counter()
    result, oof, fitted = cross_validate_scores(
        estimator,
        train_features[all_model_columns],
        train['target'],
        temporal_splits,
    )
    result.insert(0, 'model', model_name)
    candidate_results[model_name] = result
    candidate_oof[model_name] = oof
    candidate_fitted[model_name] = fitted
    print(model_name, 'completed in', round(time.perf_counter() - started, 1), 'seconds')
    display(result)

# Объединяем результаты baseline и деревьев.
model_comparison = pd.concat(
    [
        baseline_results.assign(model='baseline_logreg_27_features'),
        full_lr_results.assign(model='baseline_logreg_all_features'),
    ] + list(candidate_results.values()),
    ignore_index=True,
)
# Считаем средний Precision@Recall каждой модели.
comparison_summary = model_comparison.groupby('model').agg(
    precision_mean=('precision_at_recall', 'mean'),
    precision_std=('precision_at_recall', 'std'),
)
display(comparison_summary.sort_values('precision_mean', ascending=False))

lightgbm completed in 6.0 seconds


,model,fold,train_size,valid_size,valid_positives,precision_at_recall,threshold
0,lightgbm,fold_1,4217,2556,199,0.723077,0.278433
1,lightgbm,fold_2,6773,2367,195,0.698980,0.251188
2,lightgbm,fold_3,9140,1951,160,0.730769,0.429422


catboost completed in 27.6 seconds


,model,fold,train_size,valid_size,valid_positives,precision_at_recall,threshold
0,catboost,fold_1,4217,2556,199,0.633929,0.369867
1,catboost,fold_2,6773,2367,195,0.722513,0.507283
2,catboost,fold_3,9140,1951,160,0.727273,0.639415


hist_gradient_boosting completed in 44.1 seconds


,model,fold,train_size,valid_size,valid_positives,precision_at_recall,threshold
0,hist_gradient_boosting,fold_1,4217,2556,199,0.726804,0.435500
1,hist_gradient_boosting,fold_2,6773,2367,195,0.736559,0.464059
2,hist_gradient_boosting,fold_3,9140,1951,160,0.729032,0.637545


xgboost completed in 16.1 seconds


,model,fold,train_size,valid_size,valid_positives,precision_at_recall,threshold
0,xgboost,fold_1,4217,2556,199,0.732984,0.301355
1,xgboost,fold_2,6773,2367,195,0.717277,0.299124
2,xgboost,fold_3,9140,1951,160,0.761905,0.455778


,precision_mean,precision_std
model,,
xgboost,0.737389,0.022637
hist_gradient_boosting,0.730799,0.005112
lightgbm,0.717609,0.016585
catboost,0.694571,0.052572
baseline_logreg_all_features,0.470769,0.112134
baseline_logreg_27_features,0.280663,0.034716


### 5. Ограниченный подбор гиперпараметров Optuna

Для каждой модели создается отдельное study с `TPESampler` и фиксированным seed: он равен `RANDOM_STATE + номер модели`. Первый trial повторяет стартовую конфигурацию, следующие исследуют компактное пространство; всего выполняется **10 trials на модель**. В objective используется только средний `precision_at_recall` на трёх временных разбиениях, test не передается в Optuna.

In [30]:
# Импортируем Optuna и сборщик мусора для последовательного перебора trials.
import gc
import optuna

# Ограничиваем число trials и уменьшаем служебный вывод Optuna.
OPTUNA_TRIALS_PER_MODEL = 10
optuna.logging.set_verbosity(optuna.logging.WARNING)

def suggest_model_params(trial, model_name):
    """Задаёт пространство гиперпараметров Optuna для выбранной модели."""
    if model_name == 'lightgbm':
        return {
            'n_estimators': trial.suggest_int('n_estimators', 300, 650, step=50),
            'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.08, log=True),
            'num_leaves': trial.suggest_categorical('num_leaves', [7, 15, 31]),
            'min_child_samples': trial.suggest_int('min_child_samples', 20, 80, step=10),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.70, 1.00),
            'reg_alpha': trial.suggest_float('reg_alpha', 0.01, 3.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.3, 8.0, log=True),
            'scale_pos_weight': trial.suggest_float('scale_pos_weight', 2.0, 5.0),
        }
    if model_name == 'catboost':
        return {
            'iterations': trial.suggest_int('iterations', 300, 550, step=50),
            'depth': trial.suggest_int('depth', 4, 8),
            'learning_rate': trial.suggest_float('learning_rate', 0.025, 0.09, log=True),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 12.0, log=True),
            'random_strength': trial.suggest_float('random_strength', 0.2, 3.0, log=True),
        }
    if model_name == 'hist_gradient_boosting':
        return {
            'max_iter': trial.suggest_int('max_iter', 200, 450, step=50),
            'learning_rate': trial.suggest_float('learning_rate', 0.025, 0.10, log=True),
            'max_leaf_nodes': trial.suggest_categorical('max_leaf_nodes', [7, 15, 31]),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 15, 60, step=5),
            'l2_regularization': trial.suggest_float('l2_regularization', 0.2, 8.0, log=True),
        }
    if model_name == 'xgboost':
        return {
            'n_estimators': trial.suggest_int('n_estimators', 300, 650, step=50),
            'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.08, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 7),
            'min_child_weight': trial.suggest_float('min_child_weight', 1.0, 15.0, log=True),
            'subsample': trial.suggest_float('subsample', 0.75, 1.00),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.70, 1.00),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.5, 8.0, log=True),
            'scale_pos_weight': trial.suggest_float('scale_pos_weight', 2.0, 5.0),
        }
    raise ValueError(f'Unknown model: {model_name}')

def build_tuned_model(model_name, params):
    """Создаёт модель указанного типа с переданными гиперпараметрами."""
    if model_name == 'lightgbm':
        return LGBMClassifier(
            objective='binary',
            max_depth=-1,
            subsample=0.9,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            deterministic=True,
            force_col_wise=True,
            verbosity=-1,
            **params,
        )
    if model_name == 'catboost':
        return CatBoostClassifier(
            random_seed=RANDOM_STATE,
            auto_class_weights='Balanced',
            loss_function='Logloss',
            verbose=False,
            allow_writing_files=False,
            thread_count=-1,
            **params,
        )
    if model_name == 'hist_gradient_boosting':
        return HistGradientBoostingClassifier(
            class_weight='balanced',
            random_state=RANDOM_STATE,
            **params,
        )
    if model_name == 'xgboost':
        return XGBClassifier(
            objective='binary:logistic',
            reg_alpha=0.2,
            tree_method='hist',
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=0,
            **params,
        )
    raise ValueError(f'Unknown model: {model_name}')

# Фиксируем стартовый trial для каждой модели.
STARTING_TRIALS = {
    'lightgbm': {
        'n_estimators': 450, 'learning_rate': 0.03, 'num_leaves': 15,
        'min_child_samples': 40, 'colsample_bytree': 0.85,
        'reg_alpha': 0.2, 'reg_lambda': 2.0, 'scale_pos_weight': 3.0,
    },
    'catboost': {
        'iterations': 450, 'depth': 6, 'learning_rate': 0.04,
        'l2_leaf_reg': 5.0, 'random_strength': 1.0,
    },
    'hist_gradient_boosting': {
        'max_iter': 300, 'learning_rate': 0.05, 'max_leaf_nodes': 15,
        'min_samples_leaf': 30, 'l2_regularization': 2.0,
    },
    'xgboost': {
        'n_estimators': 450, 'learning_rate': 0.03, 'max_depth': 5,
        'min_child_weight': 5.0, 'subsample': 0.9,
        'colsample_bytree': 0.85, 'reg_lambda': 2.0,
        'scale_pos_weight': 3.0,
    },
}

def optuna_objective(trial, model_name):
    """Вычисляет средний Precision@Recall≥0.70 на временных разбиениях для trial."""
    params = suggest_model_params(trial, model_name)
    # Готовим данные и список значений основной метрики.
    fold_precision = []
    X = train_features[all_model_columns]
    y = train['target']
    # Обучаем модель заново на каждом временном разбиении.
    for _, train_idx, valid_idx in temporal_splits:
        estimator = build_tuned_model(model_name, params)
        estimator.fit(X.iloc[train_idx], y.iloc[train_idx])
        scores = estimator.predict_proba(X.iloc[valid_idx])[:, 1]
        fold_precision.append(
            precision_at_recall(y.iloc[valid_idx], scores)
        )
        del estimator
    # Сохраняем значения метрики для разбиений и возвращаем их среднее.
    trial.set_user_attr('fold_precision', [float(v) for v in fold_precision])
    gc.collect()
    return float(np.mean(fold_precision))

In [42]:
# Задаём порядок моделей и связанный с ним seed для sampler.
TUNING_MODEL_ORDER = [
    'lightgbm',
    'catboost',
    'hist_gradient_boosting',
    'xgboost',
]

# Сохраняем studies в памяти, чтобы повторный запуск не дублировал trials.
if 'optuna_studies' not in globals():
    optuna_studies = {}

# Запускаем ограниченный подбор отдельно для каждой модели.
for model_offset, model_name in enumerate(TUNING_MODEL_ORDER):
    if model_name not in optuna_studies:
        sampler = optuna.samplers.TPESampler(
            seed=RANDOM_STATE + model_offset
        )
        study = optuna.create_study(
            direction='maximize',
            sampler=sampler,
            study_name=f'{model_name}_precision_at_recall',
        )
        study.enqueue_trial(STARTING_TRIALS[model_name])
        optuna_studies[model_name] = study

    # Считаем только завершённые trials: ожидающий стартовый trial не должен уменьшать лимит.
    study = optuna_studies[model_name]
    finished_trials = sum(
        trial.state.is_finished()
        for trial in study.trials
    )
    remaining_trials = max(
        0,
        OPTUNA_TRIALS_PER_MODEL - finished_trials,
    )
    started = time.perf_counter()

    # Выполняем только недостающие trials последовательно.
    if remaining_trials:
        study.optimize(
            lambda trial, name=model_name: optuna_objective(trial, name),
            n_trials=remaining_trials,
            n_jobs=1,
            show_progress_bar=False,
            gc_after_trial=True,
        )
    print(
        model_name,
        'completed trials=', sum(
            trial.state.is_finished()
            for trial in study.trials
        ),
        'best mean precision=', round(study.best_value, 6),
        'additional time=', round(
            time.perf_counter() - started, 1
        ), 'seconds',
    )
    print('best params:', study.best_params)

# Объединяем историю всех studies в одну таблицу.
optuna_trials = pd.concat(
    [
        study.trials_dataframe().assign(model=model_name)
        for model_name, study in optuna_studies.items()
    ],
    ignore_index=True,
)
display(
    optuna_trials[
        ['model', 'number', 'value', 'state']
    ].sort_values(['model', 'value'], ascending=[True, False])
)

lightgbm completed trials= 10 best mean precision= 0.745262 additional time= 5.8 seconds
best params: {'n_estimators': 600, 'learning_rate': 0.0263432138345051, 'num_leaves': 15, 'min_child_samples': 70, 'colsample_bytree': 0.9313811040057838, 'reg_alpha': 0.015255188579331402, 'reg_lambda': 0.9733794545759303, 'scale_pos_weight': 2.347607178575389}
catboost completed trials= 10 best mean precision= 0.720028 additional time= 44.5 seconds
best params: {'iterations': 550, 'depth': 8, 'learning_rate': 0.04095681249581239, 'l2_leaf_reg': 10.71681377761754, 'random_strength': 0.6687788150873426}
hist_gradient_boosting completed trials= 10 best mean precision= 0.734068 additional time= 59.0 seconds
best params: {'max_iter': 450, 'learning_rate': 0.028909032034101143, 'max_leaf_nodes': 7, 'min_samples_leaf': 45, 'l2_regularization': 0.8548471337428305}
xgboost completed trials= 10 best mean precision= 0.748491 additional time= 14.6 seconds
best params: {'n_estimators': 300, 'learning_rate': 0

,model,number,value,state
15,catboost,5,0.720028,COMPLETE
18,catboost,8,0.720016,COMPLETE
19,catboost,9,0.704289,COMPLETE
10,catboost,0,0.694571,COMPLETE
12,catboost,2,0.688096,COMPLETE
14,catboost,4,0.685779,COMPLETE
16,catboost,6,0.682760,COMPLETE
11,catboost,1,0.677967,COMPLETE
17,catboost,7,0.674569,COMPLETE
13,catboost,3,0.662240,COMPLETE


In [43]:
# Создаём модели с лучшими параметрами каждого study.
tuned_models = {
    model_name: build_tuned_model(model_name, study.best_params)
    for model_name, study in optuna_studies.items()
}
# Готовим хранилища повторной валидации лучших конфигураций.
tuned_results = {}
tuned_oof = {}
tuned_fitted = {}

# Проверяем лучшие параметры на тех же временных разбиениях по precision_at_recall.
for model_name, estimator in tuned_models.items():
    started = time.perf_counter()
    result, oof, fitted = cross_validate_scores(
        estimator,
        train_features[all_model_columns],
        train['target'],
        temporal_splits,
    )
    result.insert(0, 'model', f'{model_name}_optuna')
    tuned_results[model_name] = result
    tuned_oof[model_name] = oof
    tuned_fitted[model_name] = fitted
    print(model_name, 'best parameters validated in', round(time.perf_counter() - started, 1), 'seconds')
    display(result)

# Собираем сводные результаты настроенных и стартовых моделей.
tuned_model_comparison = pd.concat(tuned_results.values(), ignore_index=True)
tuned_summary = tuned_model_comparison.groupby('model').agg(
    precision_mean=('precision_at_recall', 'mean'),
    precision_std=('precision_at_recall', 'std'),
)
full_model_summary = pd.concat([comparison_summary, tuned_summary])
display(full_model_summary.sort_values('precision_mean', ascending=False))

lightgbm best parameters validated in 9.2 seconds


,model,fold,train_size,valid_size,valid_positives,precision_at_recall,threshold
0,lightgbm_optuna,fold_1,4217,2556,199,0.725389,0.230984
1,lightgbm_optuna,fold_2,6773,2367,195,0.732620,0.277832
2,lightgbm_optuna,fold_3,9140,1951,160,0.777778,0.439340


catboost best parameters validated in 124.1 seconds


,model,fold,train_size,valid_size,valid_positives,precision_at_recall,threshold
0,catboost_optuna,fold_1,4217,2556,199,0.689655,0.326231
1,catboost_optuna,fold_2,6773,2367,195,0.718750,0.344976
2,catboost_optuna,fold_3,9140,1951,160,0.751678,0.534708


hist_gradient_boosting best parameters validated in 48.5 seconds


,model,fold,train_size,valid_size,valid_positives,precision_at_recall,threshold
0,hist_gradient_boosting_optuna,fold_1,4217,2556,199,0.700000,0.594775
1,hist_gradient_boosting_optuna,fold_2,6773,2367,195,0.765363,0.677859
2,hist_gradient_boosting_optuna,fold_3,9140,1951,160,0.736842,0.748703


xgboost best parameters validated in 18.7 seconds


,model,fold,train_size,valid_size,valid_positives,precision_at_recall,threshold
0,xgboost_optuna,fold_1,4217,2556,199,0.752688,0.302729
1,xgboost_optuna,fold_2,6773,2367,195,0.722513,0.254712
2,xgboost_optuna,fold_3,9140,1951,160,0.770270,0.423832


,precision_mean,precision_std
model,,
xgboost_optuna,0.748491,0.024154
lightgbm_optuna,0.745262,0.028390
xgboost,0.737389,0.022637
hist_gradient_boosting_optuna,0.734068,0.032770
hist_gradient_boosting,0.730799,0.005112
catboost_optuna,0.720028,0.031031
lightgbm,0.717609,0.016585
catboost,0.694571,0.052572
baseline_logreg_all_features,0.470769,0.112134


### 6. Выбор партнера для ансамбля

Лучший одиночный trial не обязан давать лучший ансамбль. Поэтому вместе с индивидуально лучшим LightGBM проверяется trial 4 - второй по Optuna-objective и ранее лучший партнер расширенного XGBoost. Выбор по-прежнему делается только на тех же временных разбиениях.

In [44]:
# Выбираем конкретный trial LightGBM для проверки в ансамбле.
ENSEMBLE_LGB_TRIAL_NUMBER = 4
ensemble_lgb_trial = next(
    trial
    for trial in optuna_studies['lightgbm'].trials
    if trial.number == ENSEMBLE_LGB_TRIAL_NUMBER
)
# Восстанавливаем LightGBM по параметрам выбранного trial.
ensemble_lgb_model = build_tuned_model(
    'lightgbm',
    ensemble_lgb_trial.params,
)
# Получаем его метрику и OOF-прогнозы на временных разбиениях.
ensemble_lgb_results, ensemble_lgb_oof, ensemble_lgb_fitted = (
    cross_validate_scores(
        ensemble_lgb_model,
        train_features[all_model_columns],
        train['target'],
        temporal_splits,
    )
)
ensemble_lgb_results.insert(
    0,
    'model',
    f'lightgbm_optuna_trial_{ENSEMBLE_LGB_TRIAL_NUMBER}',
)
# Выводим параметры и качество кандидата ансамбля.
print(
    'LightGBM ensemble candidate trial:',
    ENSEMBLE_LGB_TRIAL_NUMBER,
    'Optuna objective:',
    round(float(ensemble_lgb_trial.value), 6),
)
print('Parameters:', ensemble_lgb_trial.params)
display(ensemble_lgb_results)

LightGBM ensemble candidate trial: 4 Optuna objective: 0.742212
Parameters: {'n_estimators': 500, 'learning_rate': 0.02533353227775327, 'num_leaves': 31, 'min_child_samples': 70, 'colsample_bytree': 0.7913841307520112, 'reg_alpha': 0.01745603763579741, 'reg_lambda': 2.8367179456041307, 'scale_pos_weight': 3.3204574812188037}


,model,fold,train_size,valid_size,valid_positives,precision_at_recall,threshold
0,lightgbm_optuna_trial_4,fold_1,4217,2556,199,0.713568,0.229377
1,lightgbm_optuna_trial_4,fold_2,6773,2367,195,0.745946,0.236671
2,lightgbm_optuna_trial_4,fold_3,9140,1951,160,0.767123,0.449841


## 7. Дополнительные признаки и сравнение ансамблей

Сначала выбирается лучшая настроенная одиночная модель и рассматривается важность её признаков. Затем на той же временной валидации проверяется вклад дополнительных признаков и сравниваются разные комбинации моделей.

In [45]:
# Выбираем лучшую одиночную модель по средней основной метрике.
best_model_name = tuned_summary['precision_mean'].idxmax().removesuffix('_optuna')
best_results = tuned_results[best_model_name]
best_models = tuned_fitted[best_model_name]


print('Лучшая настроенная одиночная модель:', best_model_name)
display(best_results[['fold', 'threshold', 'precision_at_recall']])

Лучшая настроенная одиночная модель: xgboost


,fold,threshold,precision_at_recall
0,fold_1,0.302729,0.752688
1,fold_2,0.254712,0.722513
2,fold_3,0.423832,0.770270


In [46]:
# Берём модель последнего временного разбиения для анализа важности.
last_fold_name, _, last_valid_idx = temporal_splits[-1]
last_model = best_models[-1]

# Сортируем встроенную важность метрики и выводим топ 20.
importance = pd.DataFrame({
    'feature': all_model_columns,
    'importance': last_model.feature_importances_,
}).sort_values('importance', ascending=False)
print('built-in feature importance for', best_model_name, 'on', last_fold_name)
display(importance.head(20))

built-in feature importance for xgboost on fold_3


,feature,importance
186,events_per_active_minute_mean,0.061166
218,pointer_x_std,0.036992
207,session_events_max,0.032248
146,gap_q75,0.027460
97,item_category_dist_nunique,0.026651
216,pointer_x_count,0.026167
102,item_location_dist_nonnull_count,0.024966
166,dedup_gap_q75,0.022693
230,pointer_y_nunique,0.016058
94,item_id_dist_entropy,0.015630


In [47]:
# Строим дополнительные признаки и присоединяем их к основному набору.
additional_train_features = build_additional_features(events, train)
enhanced_train_features = train_features.merge(
    additional_train_features,
    on='cookie_id',
    how='left',
    validate='one_to_one',
)

# Определяем список и число добавленных признаков.
enhanced_columns = [c for c in enhanced_train_features.columns if c != 'cookie_id']
added_features = sorted(set(enhanced_columns) - set(all_model_columns))
print('Количество дополнительных признаков:', len(added_features))
print(added_features)

# Проверяем расширенный набор на той же модели и временных разбиениях.
enhanced_results, enhanced_oof, enhanced_models = cross_validate_scores(
    tuned_models[best_model_name],
    enhanced_train_features[enhanced_columns],
    train['target'],
    temporal_splits,
)
enhanced_results.insert(0, 'model', f'{best_model_name}_optuna_enhanced')
display(enhanced_results)

# Сравниваем исходную и расширенную версии по Precision@Recall.
before_after = pd.DataFrame({
    'version': ['основные признаки', 'основные и дополнительные признаки'],
    'precision_mean': [
        best_results.precision_at_recall.mean(),
        enhanced_results.precision_at_recall.mean(),
    ],
    'precision_last_split': [
        best_results.precision_at_recall.iloc[-1],
        enhanced_results.precision_at_recall.iloc[-1],
    ],
})
display(before_after)

Количество дополнительных признаков: 19
['pointer_bbox_area', 'pointer_distance_max', 'pointer_distance_mean', 'pointer_distance_median', 'pointer_distance_std', 'pointer_speed_max', 'pointer_speed_mean', 'pointer_x_range', 'pointer_y_range', 'pointer_zero_distance_share', 'same_item_category_adjacent_share', 'same_item_id_adjacent_share', 'search_page_diff_max', 'search_page_diff_mean', 'search_page_diff_median', 'search_page_diff_min', 'search_page_diff_std', 'search_page_forward_share', 'search_page_same_adjacent_share']


,model,fold,train_size,valid_size,valid_positives,precision_at_recall,threshold
0,xgboost_optuna_enhanced,fold_1,4217,2556,199,0.744681,0.283929
1,xgboost_optuna_enhanced,fold_2,6773,2367,195,0.748634,0.287499
2,xgboost_optuna_enhanced,fold_3,9140,1951,160,0.772414,0.449147


,version,precision_mean,precision_last_split
0,основные признаки,0.748491,0.770270
1,основные и дополнительные признаки,0.755243,0.772414


In [48]:
# Импортируем генератор пар моделей для перебора ансамблей.
from itertools import combinations

def evaluate_existing_oof(name, scores, splits=temporal_splits):
    """Считает Precision@Recall по временным разбиениям для уже полученных OOF-прогнозов."""
    # Готовим Precision@Recall отдельно для каждого validation-блока.
    rows = []
    # Выбираем OOF-прогнозы только соответствующего временного разбиения.
    for fold_name, _, valid_idx in splits:
        fold_target = train.target.iloc[valid_idx]
        fold_scores = scores[valid_idx]
        rows.append({
            'model': name,
            'fold': fold_name,
            'precision_at_recall': precision_at_recall(fold_target, fold_scores),
            'threshold': best_threshold_at_recall(fold_target, fold_scores),
        })
    # Возвращаем таблицу с метрикой по каждому временному разбиению.
    return pd.DataFrame(rows)

# Добавляем базовые равновесные смеси стартовых моделей.
blend_experiments = {
    'base_hist_lgb_equal': (
        0.5 * candidate_oof['hist_gradient_boosting']
        + 0.5 * candidate_oof['lightgbm']
    ),
    'base_hist_cat_equal': (
        0.5 * candidate_oof['hist_gradient_boosting']
        + 0.5 * candidate_oof['catboost']
    ),
}

# Перебираем попарные смеси настроенных моделей.
for left, right in combinations(TUNING_MODEL_ORDER, 2):
    blend_experiments[f'optuna_{left}_{right}_equal'] = (
        0.5 * tuned_oof[left] + 0.5 * tuned_oof[right]
    )

# Проверяем ансамбль из трёх настроенных моделей.
blend_experiments['optuna_xgb_lgb_hist_equal'] = (
    tuned_oof['xgboost']
    + tuned_oof['lightgbm']
    + tuned_oof['hist_gradient_boosting']
) / 3.0
# Проверяем XGBoost с дополнительными признаками в паре с LightGBM.
blend_experiments['optuna_lightgbm_xgboost_enhanced_equal'] = (
    0.5 * tuned_oof['lightgbm'] + 0.5 * enhanced_oof
)
# Сравниваем выбранный trial LightGBM с обычным и расширенным XGBoost.
blend_experiments['optuna_lgb_trial4_xgboost_equal'] = (
    0.5 * ensemble_lgb_oof + 0.5 * tuned_oof['xgboost']
)
blend_experiments['optuna_lgb_trial4_xgboost_enhanced_equal'] = (
    0.5 * ensemble_lgb_oof + 0.5 * enhanced_oof
)

# Считаем Precision@Recall всех ансамблей по временным разбиениям.
blend_results = pd.concat(
    [evaluate_existing_oof(name, scores) for name, scores in blend_experiments.items()],
    ignore_index=True,
)
# Агрегируем средний Precision@Recall и разброс между временными разбиениями.
blend_summary = blend_results.groupby('model').agg(
    precision_mean=('precision_at_recall', 'mean'),
    precision_std=('precision_at_recall', 'std'),
)
display(
    pd.concat([full_model_summary, blend_summary])
    .sort_values('precision_mean', ascending=False)
)

,precision_mean,precision_std
model,,
optuna_lgb_trial4_xgboost_enhanced_equal,0.773567,0.006401
optuna_lgb_trial4_xgboost_equal,0.769329,0.008786
optuna_lightgbm_xgboost_equal,0.759283,0.011372
optuna_lightgbm_xgboost_enhanced_equal,0.757912,0.012722
base_hist_cat_equal,0.754192,0.025072
optuna_hist_gradient_boosting_xgboost_equal,0.749283,0.015369
xgboost_optuna,0.748491,0.024154
optuna_catboost_xgboost_equal,0.745319,0.004247
lightgbm_optuna,0.745262,0.028390


In [49]:
# Собираем средние результаты ключевых ансамблей.
blend_mean = blend_results.groupby('model')['precision_at_recall'].mean()
selected_ensemble_name = 'optuna_lgb_trial4_xgboost_enhanced_equal'
selected_ensemble_score = float(blend_mean[selected_ensemble_name])

# Сравниваем выбранную пару с одиночными моделями и другими комбинациями.
ensemble_justification = pd.DataFrame([
    {
        'вариант': 'LightGBM trial 4',
        'число_моделей': 1,
        'precision_at_recall': ensemble_lgb_results.precision_at_recall.mean(),
    },
    {
        'вариант': 'XGBoost с дополнительными признаками',
        'число_моделей': 1,
        'precision_at_recall': enhanced_results.precision_at_recall.mean(),
    },
    {
        'вариант': 'LightGBM trial 4 + расширенный XGBoost',
        'число_моделей': 2,
        'precision_at_recall': selected_ensemble_score,
    },
    {
        'вариант': 'лучший LightGBM + XGBoost',
        'число_моделей': 2,
        'precision_at_recall': blend_mean['optuna_lightgbm_xgboost_equal'],
    },
    {
        'вариант': 'HistGradientBoosting + XGBoost',
        'число_моделей': 2,
        'precision_at_recall': blend_mean['optuna_hist_gradient_boosting_xgboost_equal'],
    },
    {
        'вариант': 'CatBoost + XGBoost',
        'число_моделей': 2,
        'precision_at_recall': blend_mean['optuna_catboost_xgboost_equal'],
    },
    {
        'вариант': 'XGBoost + LightGBM + HistGradientBoosting',
        'число_моделей': 3,
        'precision_at_recall': blend_mean['optuna_xgb_lgb_hist_equal'],
    },
])
ensemble_justification['отставание_от_выбранного'] = (
    selected_ensemble_score
    - ensemble_justification['precision_at_recall']
)
display(
    ensemble_justification.sort_values(
        'precision_at_recall',
        ascending=False,
    ).reset_index(drop=True)
)

# Сравниваем основные ансамбли отдельно на каждом временном разбиении.
models_for_split_comparison = {
    selected_ensemble_name: 'выбранная пара',
    'optuna_lightgbm_xgboost_equal': 'лучший LightGBM + XGBoost',
    'optuna_hist_gradient_boosting_xgboost_equal': 'HistGradientBoosting + XGBoost',
    'optuna_catboost_xgboost_equal': 'CatBoost + XGBoost',
    'optuna_xgb_lgb_hist_equal': 'три модели',
}
ensemble_by_split = (
    blend_results.loc[
        blend_results.model.isin(models_for_split_comparison),
        ['fold', 'model', 'precision_at_recall'],
    ]
    .pivot(
        index='fold',
        columns='model',
        values='precision_at_recall',
    )
    .rename(columns=models_for_split_comparison)
)
ensemble_by_split.index.name = 'временное_разбиение'
display(ensemble_by_split)

,вариант,число_моделей,precision_at_recall,отставание_от_выбранного
0,LightGBM trial 4 + расширенный XGBoost,2,0.773567,0.000000
1,лучший LightGBM + XGBoost,2,0.759283,0.014284
2,XGBoost с дополнительными признаками,1,0.755243,0.018325
3,HistGradientBoosting + XGBoost,2,0.749283,0.024284
4,CatBoost + XGBoost,2,0.745319,0.028248
5,XGBoost + LightGBM + HistGradientBoosting,3,0.742501,0.031067
6,LightGBM trial 4,1,0.742212,0.031355


model,CatBoost + XGBoost,HistGradientBoosting + XGBoost,выбранная пара,лучший LightGBM + XGBoost,три модели
временное_разбиение,,,,,
fold_1,0.748663,0.732984,0.774725,0.752688,0.723077
fold_2,0.740541,0.751351,0.766667,0.752747,0.752747
fold_3,0.746753,0.763514,0.779310,0.772414,0.751678


### Почему в финальном ансамбле две модели

Выбранная пара объединяет LightGBM trial 4 и XGBoost с дополнительными признаками. По отдельности они получили примерно 0.7422 и 0.7552, а после усреднения вероятностей результат вырос до 0.7736. Значит, модели совершают не полностью одинаковые ошибки и дополняют друг друга.

Добавление моделей само по себе не гарантирует улучшения. Проверенная комбинация XGBoost, LightGBM и HistGradientBoosting получила 0.7425. Пары XGBoost с HistGradientBoosting и CatBoost также оказались хуже выбранного варианта. При равном усреднении более слабая модель может изменить порядок объектов около рабочего порога и снизить precision при требуемом recall.

Поэтому финальный вариант содержит две модели: это лучший результат среди проверенных комбинаций на одинаковой временной валидации. Такой вывод не означает, что две модели теоретически являются идеальным вариантом для любых весов, но добавление остальных моделей в ансамбль не показало прироста.

In [50]:
from sklearn.inspection import permutation_importance

# Выбираем финальный ансамбль и выводим его результаты на временной валидации.
final_model_name = 'optuna_lgb_trial4_xgboost_enhanced_equal'
final_oof = blend_experiments[final_model_name]
final_cv_results = blend_results.loc[
    blend_results.model.eq(final_model_name)
].reset_index(drop=True)
print('Результаты финального ансамбля по временным разбиениям:')
display(final_cv_results)

class FittedProbabilityAverage:
    """Усредняет вероятности нескольких уже обученных моделей."""
    def __init__(self, estimator_specs):
        # Сохраняем пары модели и набора входных колонок.
        self.estimator_specs = estimator_specs

    def fit(self, X, y=None):
        # Обучение не требуется, потому что базовые модели уже fitted.
        return self

    def predict_proba(self, X):
        # Получаем вероятности всех моделей на их наборах признаков.
        probabilities = [
            model.predict_proba(X[columns])
            for model, columns in self.estimator_specs
        ]
        return np.mean(probabilities, axis=0)

# Собираем ансамбль моделей последнего временного разбиения.
last_fold_ensemble = FittedProbabilityAverage([
    (ensemble_lgb_fitted[-1], all_model_columns),
    (enhanced_models[-1], enhanced_columns),
])

def precision_recall_scorer(estimator, X, y):
    """Считает основную метрику по вероятностям модели."""
    scores = estimator.predict_proba(X)[:, 1]
    return precision_at_recall(y, scores)

# Перемешиваем признаки последнего временного разбиения и оцениваем падение основной метрики.
started = time.perf_counter()
final_permutation = permutation_importance(
    last_fold_ensemble,
    enhanced_train_features.loc[last_valid_idx, enhanced_columns],
    train.target.iloc[last_valid_idx],
    scoring=precision_recall_scorer,
    n_repeats=2,
    random_state=RANDOM_STATE,
    n_jobs=1,
)

# Собираем и сортируем значения permutation importance.
final_importance = pd.DataFrame({
    'feature': enhanced_columns,
    'importance_mean': final_permutation.importances_mean,
    'importance_std': final_permutation.importances_std,
}).sort_values('importance_mean', ascending=False)
print(
    'Расчёт permutation importance завершён за',
    round(time.perf_counter() - started, 1),
    'секунд',
)
display(final_importance.head(20))

Результаты финального ансамбля по временным разбиениям:


,model,fold,precision_at_recall,threshold
0,optuna_lgb_trial4_xgboost_enhanced_equal,fold_1,0.774725,0.283759
1,optuna_lgb_trial4_xgboost_enhanced_equal,fold_2,0.766667,0.286896
2,optuna_lgb_trial4_xgboost_enhanced_equal,fold_3,0.779310,0.413582


Расчёт permutation importance завершён за 50.0 секунд


,feature,importance_mean,importance_std
218,pointer_x_std,0.107471,0.028161
92,item_id_dist_nunique_ratio,0.042436,0.004848
104,item_location_dist_nunique_ratio,0.039754,0.008788
0,cookie_age_start_days,0.029467,0.022571
100,item_category_dist_entropy,0.028494,0.004150
226,pointer_y_std,0.027632,0.000000
161,dedup_gap_min,0.025093,0.002539
227,pointer_y_min,0.024682,0.017786
129,ua_linux_share,0.022519,0.005113
200,transition_entropy,0.018290,0.007687


## 8. Финальное обучение и submission

Финальная модель - равное среднее вероятностей LightGBM trial 4 на исходных 232 признаках и настроенного XGBoost на расширенном наборе из 251 признака. Хотя trial 8 оказался лучшим одиночным LightGBM после 10 Optuna-trials, trial 4 лучше дополняет XGBoost и даёт более высокую метрику ансамбля. Score остается непрерывной вероятностной оценкой; validation threshold к submission не применяется.

In [51]:
# Строим основные признаки test с vocabulary, полученным только из train.
final_vocab = feature_vocab
final_train_features = train_features.copy()
final_test_features = build_cookie_features(events, test, final_vocab)
final_columns = [c for c in final_train_features.columns if c != 'cookie_id']

# Проверяем одинаковый состав признаков и порядок cookie в train и test.
assert final_columns == [c for c in final_test_features.columns if c != 'cookie_id']
assert final_train_features.cookie_id.equals(train.cookie_id)
assert final_test_features.cookie_id.equals(test.cookie_id)
assert final_columns == all_model_columns

# Добавляем дополнительные признаки для train и test.
final_train_extra = additional_train_features.copy()
final_test_extra = build_additional_features(events, test)
final_train_enhanced = final_train_features.merge(
    final_train_extra,
    on='cookie_id',
    how='left',
    validate='one_to_one',
)
final_test_enhanced = final_test_features.merge(
    final_test_extra,
    on='cookie_id',
    how='left',
    validate='one_to_one',
)
final_enhanced_columns = [
    c for c in final_train_enhanced.columns if c != 'cookie_id'
]

# Проверяем одинаковый состав расширенных признаков.
assert final_enhanced_columns == [
    c for c in final_test_enhanced.columns if c != 'cookie_id'
]
assert final_enhanced_columns == enhanced_columns

# Клонируем выбранные модели и обучаем их на полном train.
final_lgb = clone(ensemble_lgb_model)
final_xgb = clone(tuned_models['xgboost'])
final_lgb.fit(final_train_features[final_columns], train.target)
final_xgb.fit(final_train_enhanced[final_enhanced_columns], train.target)

# Получаем вероятности класса 1 и усредняем модели поровну.
lgb_score = final_lgb.predict_proba(
    final_test_features[final_columns]
)[:, 1]
xgb_score = final_xgb.predict_proba(
    final_test_enhanced[final_enhanced_columns]
)[:, 1]
test_score = 0.5 * lgb_score + 0.5 * xgb_score

# Формируем submission напрямую в порядке строк test.
submission = pd.DataFrame({
    'cookie_id': test['cookie_id'].to_numpy(),
    'score': test_score,
})

# Проверяем структуру, полноту и допустимый диапазон submission.
assert list(submission.columns) == ['cookie_id', 'score']
assert len(submission) == len(test)
assert submission.cookie_id.is_unique
assert submission.cookie_id.equals(test.cookie_id)
assert submission.score.notna().all()
assert np.isfinite(submission.score).all()
assert submission.score.between(0, 1).all()

# Сохраняем проверенный submission и показываем итоговую статистику.
submission.to_csv('submission.csv', index=False)
display(submission.head())

,cookie_id,score
0,ck_315fb710a0e371e7,0.000618
1,ck_a76ee3b3e3e522fd,0.048416
2,ck_94c9a4d382689e82,0.036964
3,ck_8eaf9509ad9462a0,0.000285
4,ck_9a88a5a989cb5bc6,0.009238


### 9. Какие признаки реально использовали деревья

Модель получает все входные признаки, но фактически использованным считается признак, встретившийся хотя бы в одном split. Полная таблица со split count и нормированной долей gain сохраняется в `artifacts/tree_feature_usage.csv`. В notebook выводятся только 20 признаков с наибольшим средним gain и отдельный список признаков, которые не использовало ни одно финальное дерево.

In [52]:
# Создаём каталог для таблиц с признаками финальных моделей.
ARTIFACTS_DIR = Path('artifacts')
ARTIFACTS_DIR.mkdir(exist_ok=True)

# Получаем входные признаки и importance LightGBM.
lgb_feature_names = final_lgb.booster_.feature_name()
lgb_usage = pd.DataFrame({
    'feature': lgb_feature_names,
    'in_lightgbm_input': True,
    'lightgbm_split_count': final_lgb.booster_.feature_importance(
        importance_type='split'
    ),
    'lightgbm_total_gain': final_lgb.booster_.feature_importance(
        importance_type='gain'
    ),
})

# Получаем входные признаки и importance XGBoost.
xgb_booster = final_xgb.get_booster()
xgb_feature_names = xgb_booster.feature_names or final_enhanced_columns
xgb_split_count = xgb_booster.get_score(importance_type='weight')
xgb_total_gain = xgb_booster.get_score(importance_type='total_gain')
xgb_usage = pd.DataFrame({
    'feature': xgb_feature_names,
    'in_xgboost_input': True,
    'xgboost_split_count': [
        xgb_split_count.get(name, 0) for name in xgb_feature_names
    ],
    'xgboost_total_gain': [
        xgb_total_gain.get(name, 0.0) for name in xgb_feature_names
    ],
})

# Объединяем статистики обеих моделей по имени признака.
tree_feature_usage = pd.DataFrame({
    'feature': final_enhanced_columns,
}).merge(
    lgb_usage,
    on='feature',
    how='left',
).merge(
    xgb_usage,
    on='feature',
    how='left',
)

# Заполняем отсутствующие признаки и приводим индикаторы к bool.
tree_feature_usage['in_lightgbm_input'] = (
    tree_feature_usage['in_lightgbm_input'].fillna(False).astype(bool)
)
tree_feature_usage['in_xgboost_input'] = (
    tree_feature_usage['in_xgboost_input'].fillna(False).astype(bool)
)
# Заменяем отсутствующие counts и gain нулями.
count_columns = ['lightgbm_split_count', 'xgboost_split_count']
gain_columns = ['lightgbm_total_gain', 'xgboost_total_gain']
tree_feature_usage[count_columns + gain_columns] = (
    tree_feature_usage[count_columns + gain_columns].fillna(0)
)
# Отмечаем признаки, реально использованные в разбиениях деревьев.
tree_feature_usage['used_by_lightgbm'] = (
    tree_feature_usage['lightgbm_split_count'] > 0
)
tree_feature_usage['used_by_xgboost'] = (
    tree_feature_usage['xgboost_split_count'] > 0
)
tree_feature_usage['used_by_any_tree'] = (
    tree_feature_usage['used_by_lightgbm']
    | tree_feature_usage['used_by_xgboost']
)

# Нормируем gain отдельно внутри каждой модели.
for model_name in ['lightgbm', 'xgboost']:
    gain_column = f'{model_name}_total_gain'
    gain_sum = tree_feature_usage[gain_column].sum()
    tree_feature_usage[f'{model_name}_gain_share'] = (
        tree_feature_usage[gain_column] / gain_sum
        if gain_sum > 0 else 0.0
    )

# Усредняем доли gain и сортируем признаки по использованию.
tree_feature_usage['mean_gain_share'] = (
    tree_feature_usage['lightgbm_gain_share']
    + tree_feature_usage['xgboost_gain_share']
) / 2.0
tree_feature_usage = tree_feature_usage.sort_values(
    ['used_by_any_tree', 'mean_gain_share'],
    ascending=[False, False],
).reset_index(drop=True)

# Сохраняем полную таблицу и отдельные списки входных колонок.
tree_feature_usage.to_csv(
    ARTIFACTS_DIR / 'tree_feature_usage.csv',
    index=False,
)
pd.Series(lgb_feature_names, name='feature').to_csv(
    ARTIFACTS_DIR / 'lightgbm_input_features.csv',
    index=False,
)
pd.Series(xgb_feature_names, name='feature').to_csv(
    ARTIFACTS_DIR / 'xgboost_input_features.csv',
    index=False,
)

# Выводим число входных и реально использованных признаков.
print(
    'LightGBM: input features =', len(lgb_feature_names),
    ', used in splits =', int(tree_feature_usage.used_by_lightgbm.sum()),
)
print(
    'XGBoost: input features =', len(xgb_feature_names),
    ', used in splits =', int(tree_feature_usage.used_by_xgboost.sum()),
)
print(
    'Used by at least one model =',
    int(tree_feature_usage.used_by_any_tree.sum()),
)
print('Saved:', ARTIFACTS_DIR / 'tree_feature_usage.csv')

# Показываем только 20 лидеров, сохраняя полную таблицу в CSV.
print()
print('Top 20 features used by the final trees, sorted by mean gain share:')
display(
    tree_feature_usage.loc[tree_feature_usage.used_by_any_tree].head(20)
)

# Отдельно выводим признаки, не использованные ни одной финальной моделью.
print()
print('Input features unused by both final models:')
display(
    tree_feature_usage.loc[
        ~tree_feature_usage.used_by_any_tree,
        ['feature', 'in_lightgbm_input', 'in_xgboost_input'],
    ]
)

LightGBM: input features = 232 , used in splits = 218
XGBoost: input features = 251 , used in splits = 229
Used by at least one model = 240
Saved: artifacts\tree_feature_usage.csv

Top 20 features used by the final trees, sorted by mean gain share:


,feature,in_lightgbm_input,lightgbm_split_count,lightgbm_total_gain,in_xgboost_input,xgboost_split_count,xgboost_total_gain,used_by_lightgbm,used_by_xgboost,used_by_any_tree,lightgbm_gain_share,xgboost_gain_share,mean_gain_share
0,pointer_x_std,True,156.0,26153.935396,True,92.0,12757.832031,True,True,True,0.102167,0.103053,0.102610
1,dedup_gap_q75,True,207.0,25486.467761,True,113.0,6713.852539,True,True,True,0.099560,0.054232,0.076896
2,events_per_active_minute_mean,True,183.0,13505.063221,True,72.0,12059.106445,True,True,True,0.052756,0.097409,0.075082
3,gap_q75,True,164.0,13464.295513,True,107.0,9318.756836,True,True,True,0.052597,0.075273,0.063935
4,dedup_gap_min,True,197.0,6946.373599,True,123.0,3281.080078,True,True,True,0.027135,0.026503,0.026819
5,item_category_dist_nunique_ratio,True,141.0,6091.215669,True,78.0,3128.645508,True,True,True,0.023795,0.025272,0.024533
6,session_events_max,True,49.0,3420.523278,True,31.0,2968.544922,True,True,True,0.013362,0.023979,0.018670
7,item_category_dist_entropy,True,199.0,4308.883298,True,103.0,2514.737549,True,True,True,0.016832,0.020313,0.018573
8,item_category_dist_nunique,True,104.0,3513.281639,True,48.0,2395.569824,True,True,True,0.013724,0.019350,0.016537
9,item_id_dist_nunique_ratio,True,313.0,4017.860133,True,155.0,2125.387451,True,True,True,0.015695,0.017168,0.016432



Input features unused by both final models:


,feature,in_lightgbm_input,in_xgboost_input
240,cookie_age_le_7d,True,True
241,cookie_age_le_30d,True,True
242,cookie_age_le_90d,True,True
243,platform_norm_dist_nunique,True,True
244,user_agent_dist_nonnull_count,True,True
245,ua_headless_share,True,True
246,ua_automation_share,True,True
247,gap_count,True,True
248,hour_nonnull_count,True,True
249,search_page_count,True,True


## Итоговый отчет

- **Валидация:** используются три последовательных временных разбиения с расширяющейся обучающей выборкой, потому что test расположен строго позже train. Validation-блоки: 11-13, 14-16 и 17-19 апреля; при каждом разбиении модель обучается только на предшествующих датах. Cookie не повторяются, поэтому group split не требуется.
- **Признаки:** построены 232 основных числовых признака из событий строго внутри полуоткрытого окна: counts/shares типов событий, интервалы и их квантили, регулярность, raw/deduplicated timing, bursts 10/60/300 секунд, активные часы и минуты, sessions, переходы и серии, nunique/top-share/entropy категорий, нормализованные platform/UA, item/query/search-page/pointer статистики и возраст cookie. Для XGBoost добавлены 19 признаков движения pointer, изменения search page и соседних item/category; итого 251.
- **Baseline:** LogisticRegression на 27 простых признаках получила средний `precision_at_recall` **0.2807**. Та же модель на всех 232 признаках после feature engineering дала **0.4708**.
- **Стартовые модели:** XGBoost дал **0.7374**, HistGradientBoosting **0.7308**, LightGBM **0.7176**, CatBoost **0.6946** по `precision_at_recall`.
- **Подбор гиперпараметров:** для LightGBM, CatBoost, HistGradientBoosting и XGBoost созданы отдельные Optuna study с `TPESampler` и зафиксированным seed. Для каждой модели выполнено по **10 trials**. Первый trial повторяет стартовую конфигурацию, а остальные проверяют небольшое пространство глубины, learning rate, числа деревьев, регуляризации и параметров ограничения листьев. Objective равен среднему `precision_at_recall` на трёх временных разбиениях; test в подборе не используется.
- **Результаты подбора:** лучшие средние значения `precision_at_recall` составили **0.7485** для XGBoost, **0.7453** для LightGBM, **0.7341** для HistGradientBoosting и **0.7200** для CatBoost. Для дополнительных признаков параметры повторно не подбирались: использовался тот же настроенный XGBoost, чтобы отдельно проверить влияние нового набора признаков.
- **Дополнительные признаки:** 19 признаков повысили результат настроенного XGBoost с **0.7485** до **0.7552**. После этого расширенный XGBoost использовался при проверке ансамблей.
- **Краткий анализ ансамблей:** лучший одиночный LightGBM - trial 8, однако его ансамбль с XGBoost получил **0.7593**. LightGBM trial №4 имеет более слабый одиночный результат **0.7422**, но лучше дополняет расширенный XGBoost: их равное усреднение дало **0.7736**. Комбинация XGBoost, LightGBM и HistGradientBoosting получила **0.7425**, а пары XGBoost с HistGradientBoosting и CatBoost - **0.7493** и **0.7453**. Поэтому финальный вариант содержит две модели: это лучший результат среди проверенных комбинаций на одинаковой временной валидации.
- **Лучший результат:** `precision_at_recall` по временным разбиениям равен **0.7747 / 0.7667 / 0.7793**, среднее **0.7736**. Рабочий порог выбирается отдельно на каждом разбиении при Recall >= 0.70.
- **Выбранная модель:** среднее вероятностей LightGBM trial 4 на 232 признаках и настроенного XGBoost на 251 признаке, обученных на полном train. В submission записан непрерывный score без threshold.